# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-33: Rice iPPG (HRVCam) Exploration

### What this notebook does
Characterises the Rice iPPG dataset before anything is preprocessed, with heart-rate
variability as the question it was obtained for: what is on the disk, which of the three
per-frame clocks to trust, what the pulse oximeter actually recorded and how its samples
sit in time, how precisely beats can be timed from it, and whether the protocol's paced
deep breathing shows up in the reference as it should. This first part runs from the
recording logs and the oximeter files alone - under a megabyte per recording - so the
corpus-wide picture takes under a minute and no frame is opened. The raw frames are
examined separately.

### Dataset
Pai, Veeraraghavan, Sabharwal, *HRVCam: robust camera-based measurement of heart rate
variability* (J. Biomed. Opt. 26(2), 022707, 2021), distributed by Rice University as its
Imaging Photoplethysmography dataset. Fourteen subjects (`Subj_003`-`Subj_016`; eight male,
six female), each recorded for about two minutes in five activities: stationary with
normal breathing, reading a web page, watching a video, talking, and deep breathing paced
at 6 s in and 5 s out. One camera, a Blackfly BFLY-U3-23S6C (Sony IMX249 colour sensor) at
1280x1024 and 30 fps, gain 0, exposure set per recording, under battery-powered white LED
flood lights; a CMS50D+ finger pulse oximeter as the reference. Use requires citing the
paper, and the authors ask that participant 12's images never be used for illustration.
Redistribution is not assumed.

Per recording folder `Subj_NNN/Subj_NNN/Subj_NNN_<activity>/`:
- `cam_flea3_0/FrameNNNNN.pgm` - one raw 8-bit Bayer frame per file, 1280x1024. The colour
  PNG renderings the dataset also ships are not kept; only the raw frames are used.
- `CameraTimeLog0.txt`, `CameraCyclesLog0.txt`, `CameraFrame0.txt` - one line per frame: the
  PC's time of arrival, the camera's own clock, and the camera's frame counter.
- `Setup0.log`, `Config0.txt` - the camera settings; `window.log`, `Recorder_app.log` - the
  recorder's start, stop and oximeter events.
- `PulseOX/px1_full.mat`, `px1_partial.mat` and their `_log.txt` copies - the oximeter
  waveform with a PC time per sample. The `.pkl` copies beside them are never opened:
  unpickling a file executes code from it.

### Role in Phase 3
The one measurement of heart-rate variability from video so far, `check_hrv_from_video.py`
on MCD-rPPG, found 93% of beats but 56 ms of timing jitter against a resting RMSSD of 20-27 ms,
and closed HRV for the current model. Two parts of that harness need correcting before the
result is leaned on further: its peak finder holds a fixed 0.3 s between beats, which lets
the diastolic wave pass as a beat once the heart rate is low, and its offset search stops
at +/-0.4 s, short of the offset of up to about twenty frames (0.67 s) that MCD-rPPG
documents for its own reference. This dataset is the first in reach that was recorded for
HRV, with a reference worn throughout and a protocol that includes paced deep breathing - a
built-in positive control, since breathing at 5.5 per minute drives a large, slow swing in
heart rate that any beat-timing method should recover.

Its likely roles are evaluation ones: an HRV test set, a held-out heart-rate set, and a
breathing-rate probe at a known rate. Which of them it can carry depends first on the
reference, and that is what this part settles.

### What this notebook has to settle
1. **Is everything there?** Recordings, frames against their logs, and the oximeter files,
   before any of it is trusted.
2. **Which clock dates a frame?** Each frame carries the PC's arrival time, the camera's own
   clock and a frame counter. Beat timing needs the one that does not stall.
3. **What did the oximeter record, and when?** Its file layout, its sampling rate, whether
   samples were lost, and how to place every sample in time on the same clock as the frames.
4. **How precisely can a beat be timed from it?** A 60 Hz, 7-bit finger waveform sets a floor
   under any HRV claim built on it; the fiducial and the artefact rules decide how close to
   that floor the reference gets.
5. **Does the reference show what the protocol should produce?** HRV per activity, and the
   heart-rate swing with each paced deep breath.

No face or frame is rendered anywhere in this notebook.

## Imports & Setup

In [18]:
import ast
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.optimize import linprog
from scipy.signal import butter, find_peaks, sosfiltfilt, welch

RICE_ROOT = Path(r'F:\Rice iPPG Dataset')
README_NAME = 'Read_me_IPPG Dataset.docx'
ACTIVITIES = ('stationary', 'reading', 'watching', 'talking', 'deep')   # protocol order
FRAME_DIR = 'cam_flea3_0'
FRAME_W, FRAME_H = 1280, 1024
PGM_BYTES = FRAME_W * FRAME_H + len(b'P5\n1280 1024\n255\n')   # one 8-bit frame and its header
CAMERA_LOGS = ('CameraTimeLog0.txt', 'CameraCyclesLog0.txt', 'CameraFrame0.txt')
RECORDER_LOGS = ('Setup0.log', 'Config0.txt', 'window.log', 'Recorder_app.log')
OXIMETER_FILES = ('PulseOX/px1_full.mat', 'PulseOX/px1_partial.mat',
                  'PulseOX/px1_full_log.txt', 'PulseOX/px1_partial_log.txt')

CYCLE_WRAP_S = 128.0         # the camera clock's seconds field wraps every 128 s
OX_FS = 60.0                 # CMS50D+ waveform rate; section 3 tests it against the stamps
OX_TOP_BIT = 128             # every waveform byte arrives with its top bit set
OX_FULL_SCALE = 127          # top of the 7-bit waveform; 0 and 127 are clipped values
LOSS_LOOKAHEAD = 30          # samples (0.5 s) that must all arrive a period late to accept a loss
LOSS_TOL_S = 0.003           # slack on that test, for jitter in the floor itself
HR_MIN_BPM, HR_MAX_BPM = 35.0, 180.0   # physiological limits on a beat interval
BEAT_SPACING = 0.55          # minimum spacing between beats, as a share of the dominant period
IBI_LOCAL_TOL = 0.25         # an interval this far from the median around it is an artefact
GAP_GUARD = 2                # samples either side of a beat that must be real, not interpolated
BREATH_PERIOD_S = 6.0 + 5.0  # paced deep breathing: 6 s in, 5 s out
HR_GRID_HZ = 4.0             # the heart-rate series is resampled to this rate for its spectrum
DEMO = ('003', 'deep')       # the recording printed in detail in section 3

print('Rice iPPG root:', RICE_ROOT, '| exists:', RICE_ROOT.exists())
print('README present:', (RICE_ROOT / README_NAME).exists())
print(f'oximeter taken as {OX_FS:g} Hz until section 3 measures it | beat limits '
      f'{HR_MIN_BPM:g}-{HR_MAX_BPM:g} bpm | paced breath {BREATH_PERIOD_S:g} s = '
      f'{60 / BREATH_PERIOD_S:.2f} per minute = {1 / BREATH_PERIOD_S:.3f} Hz')

Rice iPPG root: F:\Rice iPPG Dataset | exists: True
README present: True
oximeter taken as 60 Hz until section 3 measures it | beat limits 35-180 bpm | paced breath 11 s = 5.45 per minute = 0.091 Hz


## 1. Inventory
Counts what is on the disk before any of it is trusted. A recording needs three things:
its frames, the three per-frame logs that date them, and the oximeter files. The frame count
is checked against all three logs, and every frame's size against one 8-bit 1280x1024 frame -
a test on directory entries alone, so a truncated or half-written frame is caught without
opening it. The names found in each folder are tallied too, so a file that only some
recordings carry stands out.

In [19]:
SUBJECT_DIR = re.compile(r'^Subj_(\d{3})$')
RECORDING_DIR = re.compile(r'^Subj_(\d{3})_([a-z]+)$')


def frame_census(folder):
    """Returns (frames, frames not the size of one 8-bit frame) for a recording.

    Directory entries carry the file size, so every frame is checked without opening one.
    """
    path = folder / FRAME_DIR
    if not path.is_dir():
        return 0, 0
    sizes = [entry.stat().st_size for entry in os.scandir(path) if entry.name.endswith('.pgm')]
    return len(sizes), sum(size != PGM_BYTES for size in sizes)


def count_lines(path):
    """Non-empty lines in a text log, or -1 when the file is missing."""
    if not path.exists():
        return -1
    with open(path, encoding='utf-8', errors='replace') as f:
        return sum(1 for line in f if line.strip())


def scan_recordings(root=RICE_ROOT):
    """One row per recording folder, plus a tally of the names found inside them.

    Every subject folder holds a second folder of the same name, and the recordings sit
    inside that one.
    """
    rows, names = [], {}
    for subj in sorted(p for p in root.iterdir() if p.is_dir() and SUBJECT_DIR.match(p.name)):
        inner = subj / subj.name
        if not inner.is_dir():
            continue
        for folder in sorted(p for p in inner.iterdir() if p.is_dir() and RECORDING_DIR.match(p.name)):
            subject, activity = RECORDING_DIR.match(folder.name).groups()
            frames, odd_size = frame_census(folder)
            rows.append({'subject': subject, 'activity': activity, 'folder': folder,
                         'frames': frames, 'odd_size': odd_size,
                         'time_lines': count_lines(folder / 'CameraTimeLog0.txt'),
                         'cycle_lines': count_lines(folder / 'CameraCyclesLog0.txt'),
                         'counter_lines': count_lines(folder / 'CameraFrame0.txt'),
                         'missing': [f for f in CAMERA_LOGS + RECORDER_LOGS + OXIMETER_FILES
                                     if not (folder / f).exists()]})
            for entry in list(folder.iterdir()) + list((folder / 'PulseOX').glob('*')):
                key = entry.relative_to(folder).as_posix() + ('/' if entry.is_dir() else '')
                names[key] = names.get(key, 0) + 1
    return pd.DataFrame(rows), names


recordings, names = scan_recordings()
recordings['frames_match_logs'] = ((recordings.frames == recordings.time_lines)
                                   & (recordings.time_lines == recordings.cycle_lines)
                                   & (recordings.cycle_lines == recordings.counter_lines))
n_rec = len(recordings)
expected = {(s, a) for s in recordings.subject.unique() for a in ACTIVITIES}
absent = sorted(expected - set(zip(recordings.subject, recordings.activity)))
unknown = sorted(set(recordings.activity) - set(ACTIVITIES))
incomplete = recordings.loc[recordings.missing.map(len) > 0, ['subject', 'activity', 'missing']]

print(f'subjects {recordings.subject.nunique()} | recordings {n_rec} | per activity '
      f'{recordings.activity.value_counts().reindex(ACTIVITIES, fill_value=0).to_dict()}')
print('subject/activity pairs with no folder:', absent or 'none',
      '| unexpected activity names:', unknown or 'none')
print('recordings missing a log or oximeter file:', incomplete.to_dict('records') or 'none')
print()
print(f'raw frames: {recordings.frames.sum()} in {n_rec} recordings, '
      f'{recordings.frames.sum() * PGM_BYTES / 2**30:.1f} GiB | per recording '
      f'{recordings.frames.min()}-{recordings.frames.max()}')
print(f'frames that are not one 8-bit {FRAME_W}x{FRAME_H} frame by size: {recordings.odd_size.sum()}')
print(f'frame count equal to all three per-frame logs: '
      f'{int(recordings.frames_match_logs.sum())}/{n_rec}')
mismatch = recordings.loc[~recordings.frames_match_logs,
                          ['subject', 'activity', 'frames', 'time_lines', 'cycle_lines',
                           'counter_lines']]
if len(mismatch):
    print(mismatch.to_string(index=False))
print()
everywhere = sorted(k for k, v in names.items() if v == n_rec)
print(f'names in every recording ({len(everywhere)}): {", ".join(everywhere)}')
print('names in only some recordings:', {k: v for k, v in sorted(names.items()) if v < n_rec} or 'none')

subjects 14 | recordings 70 | per activity {'stationary': 14, 'reading': 14, 'watching': 14, 'talking': 14, 'deep': 14}
subject/activity pairs with no folder: none | unexpected activity names: none
recordings missing a log or oximeter file: none

raw frames: 254352 in 70 recordings, 310.5 GiB | per recording 3387-4019
frames that are not one 8-bit 1280x1024 frame by size: 0
frame count equal to all three per-frame logs: 70/70

names in every recording (19): CameraBufferSize0.txt, CameraCyclesLog0.txt, CameraFrame0.txt, CameraInfo0.log, CameraTimeLog0.txt, Config0.txt, FlyCap0.log, OT_event.log, PulseOX/, PulseOX/px1_full.mat, PulseOX/px1_full.pkl, PulseOX/px1_full_log.txt, PulseOX/px1_partial.mat, PulseOX/px1_partial.pkl, PulseOX/px1_partial_log.txt, Recorder_app.log, Setup0.log, cam_flea3_0/, window.log
names in only some recordings: none


## 2. Camera timing: three logs, and which one to trust
Every frame is dated three ways. `CameraTimeLog0.txt` holds the PC's time when the frame
arrived, `CameraCyclesLog0.txt` the camera's own clock when it was captured - seconds that
wrap every 128 s - and `CameraFrame0.txt` the camera's frame counter. The counter shows
whether a frame was dropped, and the camera clock is immune to whatever the PC was doing;
the PC's clock is the only one the oximeter shares, so it is still needed to bring the two
together.

It is not used frame by frame, though. A time written by a busy PC is late by a variable
amount and never early, so the relation between the two clocks is the lower envelope of the
points, not a least-squares line, which a burst of late stamps drags with it. `lower_line`
fits the line that lies under every point with the least total gap above it, a small linear
programme. It recovers the offset and the drift between the clocks, and what is left above
it is how late the PC was for each frame. A frame is then dated by the camera clock, mapped
onto the PC's through that line.

In [20]:
SETTINGS = {'fps_set': r'^Frame rate is ([\d.]+) fps', 'gain_db': r'^Gain is ([\d.]+)',
            'exposure_ms': r'^Exposure is ([\d.]+)'}


def read_camera_logs(folder):
    """Returns (PC arrival times, camera clock from the first frame, frame counter), in s."""
    host = np.loadtxt(folder / 'CameraTimeLog0.txt', ndmin=1)
    cycles = np.loadtxt(folder / 'CameraCyclesLog0.txt', ndmin=1)
    with open(folder / 'CameraFrame0.txt', encoding='utf-8', errors='replace') as f:
        counter = np.array([int(re.search(r'\d+', line).group()) for line in f if line.strip()])
    step = np.diff(cycles)
    step[step < 0] += CYCLE_WRAP_S
    return host, np.concatenate([[0.0], np.cumsum(step)]), counter


def lower_line(x, y):
    """(a, b) of the line a + b*x that lies under every point with the least total gap.

    Times written by a PC are late by a variable amount and never early, so a clock
    relation is the lower envelope of the points: a line constrained to lie under every
    point is held by the promptest ones alone. With x centred, the total gap depends only
    on the line's height at the centre, so the programme maximises that height. Epoch
    seconds are around 1.5e9, so both axes are shifted near zero before solving.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x0, y0 = x.mean(), y.min()
    xc = x - x0
    sol = linprog(c=[-1.0, 0.0], A_ub=np.column_stack([np.ones_like(xc), xc]), b_ub=y - y0,
                  bounds=[(None, None), (None, None)], method='highs')
    if not sol.success:
        raise RuntimeError(f'lower envelope fit failed: {sol.message}')
    a, b = sol.x
    return float(y0 + a - b * x0), float(b)


def camera_settings(folder):
    """Frame rate, gain and exposure as the camera reported them back in Setup0.log."""
    text = (folder / 'Setup0.log').read_text(encoding='utf-8', errors='replace')
    found = {key: re.search(pattern, text, flags=re.M) for key, pattern in SETTINGS.items()}
    return {key: float(m.group(1)) if m else np.nan for key, m in found.items()}


def camera_audit(row):
    """Drops, frame rate, grid deviation, clock drift and the PC's lateness for one recording.

    `grid_max_ms` is how far the camera clock strays from a uniform grid at its own rate: the
    error a store that keeps frames as a uniform sequence with one frame rate would carry.
    """
    host, clock, counter = read_camera_logs(row.folder)
    a, b = lower_line(clock, host - clock)
    lag_ms = (host - (a + (1.0 + b) * clock)) * 1000.0
    step_ms = np.diff(clock) * 1000.0
    counter_step = np.diff(counter)
    index = np.arange(clock.size)
    grid_ms = (clock - np.polyval(np.polyfit(index, clock, 1), index)) * 1000.0
    return {'subject': row.subject, 'activity': row.activity, 'frames': len(host),
            'dropped': int((counter_step[counter_step > 0] - 1).sum()),
            'backwards': int((counter_step <= 0).sum()),
            'fps': 1000.0 / float(np.median(step_ms)),
            'step_min_ms': float(step_ms.min()), 'step_max_ms': float(step_ms.max()),
            'grid_max_ms': float(np.abs(grid_ms).max()),
            'span_s': float(clock[-1]), 'drift_ppm': b * 1e6,
            'lag_med_ms': float(np.median(lag_ms)), 'lag_p99_ms': float(np.percentile(lag_ms, 99)),
            'lag_max_ms': float(lag_ms.max()), 'worst_lag_at_s': float(clock[np.argmax(lag_ms)]),
            'lag_over_10ms': float((lag_ms > 10).mean()),
            'cam_start': a, 'cam_end': a + (1.0 + b) * float(clock[-1]),
            **camera_settings(row.folder)}


camera = pd.DataFrame([camera_audit(r) for r in recordings.itertuples()])

print(f'{len(camera)} recordings | frames dropped by the camera counter: {camera.dropped.sum()} | '
      f'counter steps that go backwards: {camera.backwards.sum()}')
print(f'frame rate on the camera clock: {camera.fps.min():.3f}-{camera.fps.max():.3f} fps | '
      f'frame interval {camera.step_min_ms.min():.2f}-{camera.step_max_ms.max():.2f} ms | '
      f'duration {camera.span_s.min():.1f}-{camera.span_s.max():.1f} s')
print(f'camera clock against a uniform grid at its own rate: worst {camera.grid_max_ms.max():.3f} ms '
      f'in any recording')
print(f'camera clock against the PC clock: {camera.drift_ppm.median():+.0f} ppm '
      f'in the median recording')
print(f'PC arrival lag above the envelope, median recording: '
      f'median {camera.lag_med_ms.median():.2f} ms, '
      f'99th percentile {camera.lag_p99_ms.median():.2f} ms, worst {camera.lag_max_ms.median():.2f} ms')
late = camera[camera.lag_max_ms > 20].sort_values('lag_max_ms', ascending=False)
print(f'recordings whose PC stamps run more than 20 ms late at some point: {len(late)}')
if len(late):
    print(late[['subject', 'activity', 'span_s', 'worst_lag_at_s', 'lag_med_ms', 'lag_p99_ms',
                'lag_max_ms', 'lag_over_10ms']].round(3).to_string(index=False))
print()
print('settings reported back by the camera: frame rate',
      camera.fps_set.round(2).value_counts().to_dict(),
      '| gain', camera.gain_db.round(2).value_counts().to_dict())
print('exposure (ms, rounded) per subject and activity:')
print(camera.assign(exposure=camera.exposure_ms.round(0).astype('Int64'))
      .pivot(index='subject', columns='activity', values='exposure')
      .reindex(columns=list(ACTIVITIES)).to_string())

70 recordings | frames dropped by the camera counter: 0 | counter steps that go backwards: 0
frame rate on the camera clock: 30.017-30.018 fps | frame interval 33.27-33.36 ms | duration 112.8-133.9 s
camera clock against a uniform grid at its own rate: worst 0.046 ms in any recording
camera clock against the PC clock: -11 ppm in the median recording
PC arrival lag above the envelope, median recording: median 0.09 ms, 99th percentile 0.63 ms, worst 1.75 ms
recordings whose PC stamps run more than 20 ms late at some point: 4
subject activity  span_s  worst_lag_at_s  lag_med_ms  lag_p99_ms  lag_max_ms  lag_over_10ms
    015  reading 122.061         117.597       0.140    1660.538    2394.587          0.031
    005  reading 119.562          35.379      74.067     103.631     104.347          0.882
    006  reading 121.394          58.865       7.826      44.429      45.671          0.452
    005 watching 120.462          28.516       8.548      25.716      26.433          0.446

settings r

## 3. The oximeter: its files, its clock, and the samples it lost
The CMS50D+ streams a plethysmographic waveform over a serial link, and the recorder stored it
in `px1_full.mat`: `pulseOxRecord`, one byte per sample with its top bit set (0-127 once 128
is removed), `pulseOxTime`, the PC's time when each sample was read, and `numPulseSample`. No
pulse rate, SpO2 or beat marker is kept, so the heart rate has to come from the waveform.
`px1_partial.mat` and the text log are checked against it.

The stamps record when the PC read each sample, not when the device took it. The reader
collects the serial stream in bursts - intervals of 20, 20 and 10 ms where the device delivers
one sample every 16.7 ms - and it also loses samples, one or two at a time. Both matter for
HRV. Dating samples by their stamps adds the reading jitter to every beat time; dating them by
their index alone, at 60 Hz, puts every sample after a loss one period early, and the error
accumulates over the recording.

The device's own clock can be rebuilt because it is regular and the losses are whole samples.
Against a 60 Hz grid, the stamps minus their slot times form a floor of prompt reads with late
ones above it, and that floor steps up by exactly one period at each lost sample.
`sample_slots` walks the stamps and accepts a loss only when every one of the next half second
of samples arrives at least one period (less 3 ms) above the floor of the half second before
it, so a single late read never passes for a loss. A step down - a sample apparently taken
before its predecessor - is impossible under this model; it is counted rather than repaired,
and a count above zero would mean the model does not fit that recording. `lower_line` then
fits the floor against the slots, which gives every sample a time on the PC's clock and the
device's true rate; the stamp minus that time is the read delay, zero or more by construction.

The rebuilt times also say how much of each recording the reference covers. Three recorder
logs contain `Cannot read pulse-ox`: the oximeter link failed before the camera stopped.

A positive control closes the section. Known losses are injected into a clean 60 Hz grid at
1%, three times the dataset's own rate, each recording's rebuilt read delays are added back,
and the rule is asked to find them. A loss placed a sample or two early or late only misdates
samples next to the interpolated slot, whose beats section 4 drops anyway; what would matter is
a misplacement further away, or a count that comes out wrong. The delays come from the rebuild
itself, so this tests the rule against the jitter it actually faces, not against an
independent truth.

In [21]:
def read_oximeter(folder):
    """Returns (waveform bytes, PC read times in s, the .mat contents) from px1_full.mat."""
    mat = sio.loadmat(folder / 'PulseOX' / 'px1_full.mat', squeeze_me=True)
    values = np.atleast_1d(mat['pulseOxRecord']).astype(np.int64)
    stamps = np.atleast_1d(mat['pulseOxTime']).astype(np.float64)
    return values, stamps, mat


def oximeter_files(folder, values, stamps, mat):
    """Whether the full record agrees with its count, its text log and the partial copy."""
    part = sio.loadmat(folder / 'PulseOX' / 'px1_partial.mat', squeeze_me=True)
    part_values = np.atleast_1d(part['pulseOxRecord']).astype(np.int64)
    part_stamps = np.atleast_1d(part['pulseOxTime']).astype(np.float64)
    # The text log is a Python list literal; literal_eval parses literals and runs nothing.
    text = (folder / 'PulseOX' / 'px1_full_log.txt').read_text(encoding='utf-8', errors='replace')
    log = np.asarray(ast.literal_eval(text.strip()), dtype=np.float64)
    n_part = len(part_values)
    return {'variables': ' '.join(sorted(k for k in mat if not k.startswith('__'))),
            'count_matches': int(mat['numPulseSample']) == len(values),
            'log_matches': (len(log) == len(stamps)
                            and bool(np.allclose(log, stamps, rtol=0, atol=1e-6))),
            'partial_is_prefix': (n_part <= len(values)
                                  and bool(np.array_equal(part_values, values[:n_part]))
                                  and bool(np.allclose(part_stamps, stamps[:n_part],
                                                       rtol=0, atol=1e-6))),
            'partial_shorter_by': len(values) - n_part,
            'top_bit_set': bool(np.all((values >= OX_TOP_BIT) & (values < 2 * OX_TOP_BIT)))}


def sample_slots(stamps, fs=OX_FS, lookahead=LOSS_LOOKAHEAD, tol=LOSS_TOL_S):
    """Places every read sample in its slot on the device's own grid.

    Returns (slots, steps_down): slots[k] is k plus the samples lost before sample k. A
    stamp minus its slot time sits on a floor of prompt reads, with late reads above it,
    and the floor steps up one period at each lost sample. A loss is accepted only when all
    of the next `lookahead` samples sit at least a period (less `tol`) above the floor of the
    `lookahead` before them, so one late read never passes for a loss. A step down is
    impossible under this model; the number of such events is returned, not repaired.
    """
    period = 1.0 / fs
    n = stamps.size
    slots = np.empty(n, dtype=np.int64)
    floor = np.empty(n)
    below = np.zeros(n, dtype=bool)
    lost = 0
    for k in range(n):
        hi = min(n, k + lookahead)
        ahead = stamps[k:hi] - (np.arange(k, hi) + lost) * period
        before = ahead.min() if k == 0 else floor[max(0, k - lookahead):k].min()
        while hi - k >= 5 and ahead.min() - before >= period - tol:
            lost += 1
            ahead = ahead - period
        below[k] = ahead.min() - before <= -(period - tol)
        slots[k] = k + lost
        floor[k] = stamps[k] - slots[k] * period
    steps_down = int(below[0]) + int((below[1:] & ~below[:-1]).sum())
    return slots, steps_down


def rebuild_clock(stamps, fs=OX_FS):
    """Slots, sample times on the PC clock, read delays and the device's measured rate."""
    slots, steps_down = sample_slots(stamps, fs)
    grid_s = slots / fs
    a, b = lower_line(grid_s, stamps - grid_s)
    times = a + (1.0 + b) * grid_s
    return {'slots': slots, 'times': times, 'delay': stamps - times,
            'rate_hz': fs / (1.0 + b), 'steps_down': steps_down}


def oximeter_audit(row, cam):
    """File checks, clock rebuild and coverage of one recording: (audit row, reference)."""
    values, stamps, mat = read_oximeter(row.folder)
    clock = rebuild_clock(stamps)
    gaps = np.diff(clock['slots']) - 1
    index = np.arange(stamps.size)
    straight = np.polyval(np.polyfit(index, stamps, 1), index)
    window = (row.folder / 'window.log').read_text(encoding='utf-8', errors='replace')
    audit = {'subject': row.subject, 'activity': row.activity, 'samples': stamps.size,
             **oximeter_files(row.folder, values, stamps, mat),
             'span_s': float(stamps[-1] - stamps[0]),
             'stamp_rate_hz': (stamps.size - 1) / float(stamps[-1] - stamps[0]),
             'device_ppm': (clock['rate_hz'] / OX_FS - 1.0) * 1e6,
             'lost': int(gaps.sum()), 'loss_events': int((gaps > 0).sum()),
             'longest_gap': int(gaps.max()) if gaps.size else 0,
             'lost_share': float(gaps.sum() / (clock['slots'][-1] + 1)),
             'steps_down': clock['steps_down'],
             'delay_med_ms': float(np.median(clock['delay']) * 1000.0),
             'delay_p95_ms': float(np.percentile(clock['delay'], 95) * 1000.0),
             'delay_max_ms': float(clock['delay'].max() * 1000.0),
             'index_drift_s': float((stamps[-1] - stamps[0]) - (stamps.size - 1) / OX_FS),
             'stamp_wander_ms': float(np.ptp(stamps - straight) * 1000.0),
             'read_failures': window.count('Cannot read pulse-ox'),
             'starts_after_camera_s': float(clock['times'][0] - cam.cam_start),
             'ends_before_camera_s': float(cam.cam_end - clock['times'][-1])}
    return audit, (values - OX_TOP_BIT, clock['slots'], clock['times'])


REFERENCE = {}
rows = []
for r in recordings.itertuples():
    cam = camera.loc[(camera.subject == r.subject) & (camera.activity == r.activity)].iloc[0]
    audit, reference = oximeter_audit(r, cam)
    REFERENCE[(r.subject, r.activity)] = reference
    rows.append(audit)
oximeter = pd.DataFrame(rows)
n_ox = len(oximeter)

print('variables in px1_full.mat:', oximeter.variables.value_counts().to_dict())
print(f'numPulseSample equals the record length: {int(oximeter.count_matches.sum())}/{n_ox} | '
      f'the text log equals pulseOxTime: {int(oximeter.log_matches.sum())}/{n_ox} | '
      f'top bit set in every byte: {int(oximeter.top_bit_set.sum())}/{n_ox}')
print(f'px1_partial.mat is a leading part of px1_full.mat: '
      f'{int(oximeter.partial_is_prefix.sum())}/{n_ox}, '
      f'shorter in {int((oximeter.partial_shorter_by > 0).sum())} (by up to '
      f'{oximeter.partial_shorter_by.max()} samples), so the full record is the complete one')
print()
print(f'samples per recording {oximeter.samples.min()}-{oximeter.samples.max()} over '
      f'{oximeter.span_s.min():.1f}-{oximeter.span_s.max():.1f} s; samples per second of stamps '
      f'{oximeter.stamp_rate_hz.min():.2f}-{oximeter.stamp_rate_hz.max():.2f}')
print(f'device rate on the rebuilt clock: {OX_FS:g} Hz {oximeter.device_ppm.median():+.0f} ppm in the '
      f'median recording (interquartile {oximeter.device_ppm.quantile(0.25):+.0f} to '
      f'{oximeter.device_ppm.quantile(0.75):+.0f})')
print(f'lost samples: {oximeter.lost.sum()} in {int((oximeter.lost > 0).sum())}/{n_ox} recordings | '
      f'per recording median {oximeter.lost.median():.0f}, '
      f'range {oximeter.lost.min()}-{oximeter.lost.max()} | '
      f'every loss a single sample in {int((oximeter.longest_gap <= 1).sum())} recordings, '
      f'longest gap {oximeter.longest_gap.max()} samples')
print(f'steps down, impossible under the loss model: {oximeter.steps_down.sum()}')
print(f'read delay above the rebuilt clock, median recording: '
      f'median {oximeter.delay_med_ms.median():.1f} ms, '
      f'95th percentile {oximeter.delay_p95_ms.median():.1f} ms')
drift, wander = oximeter.index_drift_s, oximeter.stamp_wander_ms
print(f'dating samples by index alone ends {drift.median():.2f} s early in the median recording, '
      f'up to {drift.max():.2f} s; a straight line through the stamps leaves {wander.median():.0f} ms '
      f'of wander peak to peak (up to {wander.max():.0f} ms)')
_, demo_slots, demo_times = REFERENCE[DEMO]
demo_gaps = np.diff(demo_slots) - 1
demo_losses = [f'{demo_times[i + 1] - demo_times[0]:.1f} s'
               + (f' ({demo_gaps[i]} samples)' if demo_gaps[i] > 1 else '')
               for i in np.flatnonzero(demo_gaps)]
print(f'{DEMO[0]}/{DEMO[1]}: losses at ' + ', '.join(demo_losses))
print()
weak = oximeter[(oximeter.lost_share > 0.01) | (oximeter.longest_gap > 3)
                | (oximeter.delay_med_ms > 20) | (oximeter.read_failures > 0)]
print('recordings with more than 1% lost, a gap over 3 samples, a median read delay over 20 ms, '
      'or a failed oximeter read:')
print(weak[['subject', 'activity', 'lost', 'lost_share', 'longest_gap', 'delay_med_ms', 'read_failures',
            'ends_before_camera_s']].round(3).to_string(index=False))
print()
print(f'the reference starts {oximeter.starts_after_camera_s.min():.2f}-'
      f'{oximeter.starts_after_camera_s.max():.2f} s '
      f'after the first frame and ends {oximeter.ends_before_camera_s.min():+.2f} to '
      f'{oximeter.ends_before_camera_s.max():+.2f} s before the last one')
uncovered = oximeter[oximeter.ends_before_camera_s > 1]
print('recordings whose last second or more of frames has no reference:',
      uncovered[['subject', 'activity', 'ends_before_camera_s', 'read_failures']]
      .round(2).to_dict('records') or 'none')

variables in px1_full.mat: {'numPulseSample pulseOxRecord pulseOxTime': 70}
numPulseSample equals the record length: 70/70 | the text log equals pulseOxTime: 70/70 | top bit set in every byte: 70/70
px1_partial.mat is a leading part of px1_full.mat: 70/70, shorter in 12 (by up to 265 samples), so the full record is the complete one

samples per recording 5692-8002 over 112.0-133.7 s; samples per second of stamps 47.61-59.95
device rate on the rebuilt clock: 60 Hz -15 ppm in the median recording (interquartile -17 to +16)
lost samples: 2803 in 70/70 recordings | per recording median 20, range 6-1482 | every loss a single sample in 62 recordings, longest gap 14 samples
steps down, impossible under the loss model: 0
read delay above the rebuilt clock, median recording: median 5.3 ms, 95th percentile 9.8 ms
dating samples by index alone ends 0.32 s early in the median recording, up to 24.69 s; a straight line through the stamps leaves 122 ms of wander peak to peak (up to 7820 ms)
003/deep:

In [22]:
CONTROL_LOSS_RATE = 0.01   # injected losses per sample; the dataset's own rate is about 0.3%
CONTROL_RUNS = 3           # repetitions per recording, each with fresh loss positions


def recovery_check(delays, rng, loss_rate=CONTROL_LOSS_RATE):
    """Known losses on a clean grid plus a recording's own read delays, then the rule again.

    Returns (losses injected, misplacements further than GAP_GUARD samples from any true
    loss, error in the final count).
    """
    gaps = (rng.random(delays.size - 1) < loss_rate).astype(np.int64)
    truth = np.concatenate([[0], np.cumsum(1 + gaps)])
    stamps = np.maximum.accumulate(truth / OX_FS + delays)
    slots, _ = sample_slots(stamps)
    error = slots - truth
    changes = np.flatnonzero(np.diff(error)) + 1
    loss_at = np.flatnonzero(gaps) + 1
    if changes.size and loss_at.size:
        far = np.min(np.abs(changes[:, None] - loss_at[None, :]), axis=1) > GAP_GUARD
    else:
        far = np.ones(changes.size, dtype=bool)
    return loss_at.size, int(far.sum()), abs(int(error[-1]))


rng = np.random.default_rng(0)
rows = []
for r in recordings.itertuples():
    _, slots, times = REFERENCE[(r.subject, r.activity)]
    delays = read_oximeter(r.folder)[1] - times
    runs = np.array([recovery_check(delays, rng) for _ in range(CONTROL_RUNS)])
    rows.append({'subject': r.subject, 'activity': r.activity, 'injected': int(runs[:, 0].sum()),
                 'misplaced': int(runs[:, 1].sum()), 'count_error': int(runs[:, 2].sum())})
control = pd.DataFrame(rows).merge(oximeter[['subject', 'activity', 'delay_med_ms', 'delay_p95_ms']],
                                   on=['subject', 'activity'])
control['error_rate'] = (control.misplaced + control.count_error) / control.injected
print(f'positive control: {control.injected.sum()} losses injected at {CONTROL_LOSS_RATE:.0%}, '
      f'{CONTROL_RUNS} runs into each of {len(control)} recordings, '
      f'each run with the recording\'s own read delays')
print(f'final count wrong in {int((control.count_error > 0).sum())} recordings (summed error '
      f'{control.count_error.sum()}) | misplaced more than {GAP_GUARD} samples from any true loss: '
      f'{control.misplaced.sum()}, '
      f'{control.misplaced.sum() / (len(control) * CONTROL_RUNS):.2f} per run')
shaky = control[control.error_rate > 0.02].sort_values('error_rate', ascending=False)
print('recordings where the rule mishandled more than 2% of the injected losses, '
      'with their read delays:')
print(shaky[['subject', 'activity', 'misplaced', 'count_error', 'error_rate', 'delay_med_ms',
             'delay_p95_ms']].round(3).to_string(index=False) if len(shaky) else '  none')

positive control: 15082 losses injected at 1%, 3 runs into each of 70 recordings, each run with the recording's own read delays
final count wrong in 28 recordings (summed error 65) | misplaced more than 2 samples from any true loss: 59, 0.28 per run
recordings where the rule mishandled more than 2% of the injected losses, with their read delays:
subject   activity  misplaced  count_error  error_rate  delay_med_ms  delay_p95_ms
    006    reading         15           14       0.133        22.573        70.863
    006 stationary          6            5       0.065        18.020        25.797
    005 stationary          5            6       0.045        10.324        38.533
    005    reading          1            7       0.041        77.698       108.308
    015   watching          4            3       0.033        11.191        22.505
    005   watching          2            4       0.031        13.447        44.360
    013   watching          4            1       0.021         6.318   

## 4. Reference beats from a 60 Hz, 7-bit finger PPG
The waveform goes onto the rebuilt 60 Hz grid, with lost samples interpolated and marked.
Three choices then decide how precise the beat times are.

**The fiducial.** Two are measured: the steepest point of the upstroke, and the systolic peak.
The upstroke is the sharpest feature of a finger pulse and the least changed by the reflected
wave; the peak is blunt, and where the diastolic wave rises to nearly the same height the
maximum can move from one to the other between beats. Both are refined between samples with
the three-point parabola the HRV scripts already use.

**The minimum spacing between beats.** `beat_times` in `check_hrv_sampling_ceiling.py` holds a
fixed 0.3 s, which lets the diastolic wave pass as a second beat once the heart rate is low
enough. Here the spacing is 55% of the recording's own dominant beat period, read from its
spectrum, which still admits the shortest intervals of a deep-breathing swing.

**What counts as an artefact.** An interval is dropped when it lies outside 35-180 bpm, when
either of its beats has an interpolated sample, or a clipped one at 0 or 127, within two
samples of its upstroke, or when it differs by more than 25% from the median of the seven
intervals centred on it. Successive differences are taken only between two kept intervals that
are neighbours, so a single artefact cannot manufacture a large difference.

The precision floor comes from simulation: a pulse train with known beat times, sampled at
60 Hz with a random phase, rounded to whole levels at the beat amplitudes this dataset shows,
and timed with the same filters and parabola. Its timing error converts to RMSSD in
quadrature. A successive difference of intervals combines three beat times,
t[i] - 2t[i+1] + t[i+2], so independent timing errors with standard deviation s add a noise
RMSSD of q = sqrt(6)·s, and a true RMSSD r is measured as sqrt(r² + q²). The error is the
quadrature difference sqrt(r² + q²) - r; the sampling-ceiling script's `predicted` column
reports q itself, which overstates it.

In [23]:
BEAT_BAND = butter(2, [0.5, 8.0], btype='band', fs=OX_FS, output='sos')
HR_BAND = butter(2, [HR_MIN_BPM / 60.0, HR_MAX_BPM / 60.0], btype='band', fs=OX_FS, output='sos')


def on_grid(wave, slots):
    """The waveform on the device's 60 Hz grid, with lost samples interpolated and marked."""
    grid = np.full(int(slots[-1]) + 1, np.nan)
    grid[slots] = wave
    missing = np.isnan(grid)
    if missing.any():
        idx = np.arange(grid.size)
        grid[missing] = np.interp(idx[missing], idx[~missing], grid[~missing])
    return grid, missing


def dominant_period(grid, fs=OX_FS):
    """The recording's dominant beat period in samples, from a 16 s Welch spectrum."""
    f, p = welch(sosfiltfilt(HR_BAND, grid - grid.mean()), fs=fs, nperseg=int(16 * fs))
    band = (f >= HR_MIN_BPM / 60.0) & (f <= HR_MAX_BPM / 60.0)
    return fs / f[band][np.argmax(p[band])]


def refine(y, i):
    """Sub-sample position of a maximum from a parabola through three points.

    The same estimator as `refine` in check_hrv_sampling_ceiling.py: the index comes back
    unchanged at an edge, on a flat top, or when the vertex would leave the sample's cell.
    """
    if i <= 0 or i >= len(y) - 1:
        return float(i)
    left, centre, right = float(y[i - 1]), float(y[i]), float(y[i + 1])
    denominator = left - 2.0 * centre + right
    if abs(denominator) < 1e-12:
        return float(i)
    offset = 0.5 * (left - right) / denominator
    return float(i) + (offset if abs(offset) <= 0.5 else 0.0)


def find_beats(grid, missing, fs=OX_FS):
    """One row per beat: steepest-upstroke and systolic-peak times, amplitude and flags.

    A candidate must climb at least 40% as steeply as the median of the candidates around
    it, which drops ripples on the downstroke and noise between beats without a fixed
    threshold that the breathing-driven amplitude swing would defeat.
    """
    x = sosfiltfilt(BEAT_BAND, grid - grid.mean())
    slope = np.gradient(x)
    period = dominant_period(grid, fs)
    idx, _ = find_peaks(slope, distance=max(1, int(BEAT_SPACING * period)))
    if idx.size < 3:
        return pd.DataFrame(columns=['slope_s', 'peak_s', 'amplitude', 'near_gap', 'clipped']), period
    local = pd.Series(slope[idx]).rolling(9, center=True, min_periods=3).median().to_numpy()
    idx = idx[slope[idx] >= 0.4 * local]
    rows = []
    for i in idx:
        top = min(grid.size - 1, i + int(0.45 * period))
        j = i + int(np.argmax(x[i:top + 1]))
        around = grid[max(0, i - GAP_GUARD):i + GAP_GUARD + 1]
        trough = grid[max(0, i - int(0.4 * period)):i + 1].min()
        rows.append({'slope_s': refine(slope, int(i)) / fs, 'peak_s': refine(x, j) / fs,
                     'amplitude': float(grid[i:top + 1].max() - trough),
                     'near_gap': bool(missing[max(0, i - GAP_GUARD):i + GAP_GUARD + 1].any()),
                     'clipped': bool(((around <= 0) | (around >= OX_FULL_SCALE)).any())})
    return pd.DataFrame(rows), period


def intervals(beats):
    """Beat-to-beat intervals for both fiducials, with the artefact rules applied.

    `kept` marks the intervals HRV is computed from; `reason` says why the others were
    dropped, in the order the rules are applied.
    """
    t = beats.slope_s.to_numpy()
    ibi = np.diff(t)
    flagged = (beats.near_gap | beats.clipped).to_numpy()
    in_range = (ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)
    clean = ~flagged[:-1] & ~flagged[1:]
    candidate = in_range & clean
    local = (pd.Series(np.where(candidate, ibi, np.nan))
             .rolling(7, center=True, min_periods=3).median().to_numpy())
    consistent = np.abs(ibi - local) <= IBI_LOCAL_TOL * local
    return pd.DataFrame({'t': t[1:], 'ibi': ibi, 'ibi_peak': np.diff(beats.peak_s.to_numpy()),
                         'kept': candidate & consistent,
                         'reason': np.select([~in_range, ~clean, ~consistent],
                                             ['range', 'gap or clip', 'local'], default='')})


def hrv_metrics(table, column='ibi'):
    """Mean heart rate, SDNN, RMSSD and pNN50 from the kept intervals of one fiducial."""
    ibi = table[column].to_numpy()
    kept = table.kept.to_numpy()
    pair = kept[1:] & kept[:-1]
    diffs = np.diff(ibi)[pair]
    values = ibi[kept]
    if values.size < 8 or diffs.size < 8:
        return dict.fromkeys(('hr_bpm', 'sdnn_ms', 'rmssd_ms', 'pnn50'), np.nan)
    return {'hr_bpm': 60.0 / values.mean(), 'sdnn_ms': values.std(ddof=1) * 1000.0,
            'rmssd_ms': float(np.sqrt(np.mean(diffs ** 2)) * 1000.0),
            'pnn50': float(np.mean(np.abs(diffs) > 0.05))}


BEATS = {}
rows = []
for (subject, activity), (wave, slots, _) in REFERENCE.items():
    grid, missing = on_grid(wave.astype(np.float64), slots)
    beats, period = find_beats(grid, missing)
    table = intervals(beats)
    BEATS[(subject, activity)] = table
    rows.append({'subject': subject, 'activity': activity,
                 'hr_spectral_bpm': 60.0 * OX_FS / period, 'beats': len(beats),
                 'intervals': len(table), 'kept_share': float(table.kept.mean()),
                 'drop_range': int((table.reason == 'range').sum()),
                 'drop_gap_clip': int((table.reason == 'gap or clip').sum()),
                 'drop_local': int((table.reason == 'local').sum()),
                 'amplitude': float(beats.amplitude.median()),
                 'at_floor': float((wave == 0).mean()),
                 'at_ceiling': float((wave == OX_FULL_SCALE).mean()),
                 'rmssd_slope_ms': hrv_metrics(table)['rmssd_ms'],
                 'rmssd_peak_ms': hrv_metrics(table, 'ibi_peak')['rmssd_ms']})
beat_audit = pd.DataFrame(rows)

print(f'waveform at the floor of its range (0): median {beat_audit.at_floor.median():.2%} of samples, '
      f'worst {beat_audit.at_floor.max():.2%} | at the ceiling ({OX_FULL_SCALE}): median '
      f'{beat_audit.at_ceiling.median():.2%}, worst {beat_audit.at_ceiling.max():.2%}')
print(f'beat amplitude, recording medians: '
      f'{beat_audit.amplitude.min():.0f}-{beat_audit.amplitude.max():.0f} '
      f'levels of {OX_FULL_SCALE} (median {beat_audit.amplitude.median():.0f})')
print(f'beats per recording {beat_audit.beats.min()}-{beat_audit.beats.max()} | intervals kept: median '
      f'{beat_audit.kept_share.median():.1%}, lowest {beat_audit.kept_share.min():.1%}')
print(f'dropped, all recordings: outside {HR_MIN_BPM:g}-{HR_MAX_BPM:g} bpm '
      f'{beat_audit.drop_range.sum()}, '
      f'near a lost sample or clipped {beat_audit.drop_gap_clip.sum()}, '
      f'more than {IBI_LOCAL_TOL:.0%} from the local median {beat_audit.drop_local.sum()}')
print('\nfewest intervals kept, worst 6:')
print(beat_audit.nsmallest(6, 'kept_share')[['subject', 'activity', 'beats', 'kept_share',
                                             'drop_gap_clip', 'drop_local', 'at_floor']]
      .round(3).to_string(index=False))

waveform at the floor of its range (0): median 0.47% of samples, worst 8.15% | at the ceiling (127): median 0.00%, worst 0.57%
beat amplitude, recording medians: 35-65 levels of 127 (median 52)
beats per recording 101-226 | intervals kept: median 96.7%, lowest 50.3%
dropped, all recordings: outside 35-180 bpm 11, near a lost sample or clipped 499, more than 25% from the local median 24

fewest intervals kept, worst 6:
subject   activity  beats  kept_share  drop_gap_clip  drop_local  at_floor
    006 stationary    160       0.503             76           3     0.003
    005    talking    201       0.860             28           0     0.048
    008   watching    110       0.862             12           2     0.026
    004       deep    218       0.866             26           2     0.081
    013   watching    182       0.873             23           0     0.031
    014   watching    133       0.879             15           1     0.018


In [24]:
def beat_template(t):
    """A finger-PPG beat on a 0-1 scale: the systolic wave and a smaller diastolic one."""
    return (np.exp(-0.5 * ((t - 0.16) / 0.055) ** 2)
            + 0.45 * np.exp(-0.5 * ((t - 0.40) / 0.08) ** 2))


def timing_error_ms(amplitude, quantise, sub_sample, n_beats=400, seed=0, fs=OX_FS):
    """Standard deviation of the steepest-upstroke time on a simulated 60 Hz pulse train.

    The train is built on a continuous time axis, sampled with a random phase, optionally
    rounded to whole levels, and timed exactly as the reference is. The constant offset
    between a beat's onset and its steepest point is removed, so what remains is the
    sampling and rounding error alone.
    """
    rng = np.random.default_rng(seed)
    onsets = 1.0 + np.cumsum(0.8 + 0.05 * rng.standard_normal(n_beats))
    phase = rng.uniform(0.0, 1.0 / fs)
    t = np.arange(phase, onsets[-1] + 2.0, 1.0 / fs)
    wave = np.zeros_like(t)
    for onset in onsets:
        m = (t >= onset - 0.1) & (t < onset + 1.0)
        wave[m] += beat_template(t[m] - onset)
    wave = 10.0 + amplitude * wave
    if quantise:
        wave = np.round(wave)
    slope = np.gradient(sosfiltfilt(BEAT_BAND, wave - wave.mean()))
    idx, _ = find_peaks(slope, distance=int(BEAT_SPACING * 0.8 * fs))
    idx = idx[slope[idx] >= 0.4 * np.median(slope[idx])]
    est = np.array([refine(slope, int(i)) if sub_sample else float(i) for i in idx]) / fs + phase
    onset = onsets[np.clip(np.searchsorted(onsets, est - 0.4), 0, onsets.size - 1)]
    err = est - onset
    err = err[np.abs(err - np.median(err)) < 0.1]
    return float(np.std(err - np.median(err)) * 1000.0)


levels = sorted({int(round(beat_audit.amplitude.quantile(0.1))),
                 int(round(beat_audit.amplitude.median()))})
rmssd_points = [float(beat_audit.rmssd_slope_ms.quantile(q)) for q in (0.1, 0.5)]
rows = []
for amplitude in levels:
    for quantise in (False, True):
        for sub_sample in (False, True):
            sd = timing_error_ms(amplitude, quantise, sub_sample)
            q = np.sqrt(6.0) * sd
            rows.append({'amplitude': amplitude, 'rounded_to_levels': quantise,
                         'sub_sample': sub_sample,
                         'timing_sd_ms': sd, 'noise_rmssd_ms': q,
                         **{f'error_at_{r:.0f}ms': np.sqrt(r ** 2 + q ** 2) - r for r in rmssd_points}})
floor = pd.DataFrame(rows)

print(f'timing floor on a simulated 60 Hz pulse train (beat amplitudes: 10th percentile and median of '
      f'this dataset); RMSSD error evaluated at the 10th percentile and median reference RMSSD:')
print(floor.round(2).to_string(index=False))
print(f'rounding to the nearest sample alone: h/sqrt(12) = {1000 / OX_FS / np.sqrt(12):.2f} ms')
print()
gap = beat_audit.rmssd_peak_ms - beat_audit.rmssd_slope_ms
print(f'RMSSD from the systolic peak minus RMSSD from the steepest upstroke: '
      f'median {gap.median():+.1f} ms '
      f'(interquartile {gap.quantile(0.25):+.1f} to {gap.quantile(0.75):+.1f})')
print('recordings where the peak fiducial adds more than 10 ms:')
print(beat_audit.loc[gap > 10, ['subject', 'activity', 'rmssd_slope_ms', 'rmssd_peak_ms']]
      .round(1).to_string(index=False) if (gap > 10).any() else '  none')

timing floor on a simulated 60 Hz pulse train (beat amplitudes: 10th percentile and median of this dataset); RMSSD error evaluated at the 10th percentile and median reference RMSSD:
 amplitude  rounded_to_levels  sub_sample  timing_sd_ms  noise_rmssd_ms  error_at_18ms  error_at_41ms
        43              False       False          4.76           11.65           3.49           1.63
        43              False        True          0.16            0.39           0.00           0.00
        43               True       False          4.79           11.73           3.53           1.65
        43               True        True          0.49            1.20           0.04           0.02
        52              False       False          4.76           11.65           3.49           1.63
        52              False        True          0.16            0.39           0.00           0.00
        52               True       False          4.76           11.66           3.49           1.63
  

## 5. Heart-rate variability per activity
The reference's own HRV, from the kept intervals of the steepest-upstroke fiducial: mean heart
rate, SDNN, RMSSD and pNN50. Two cautions shape what is reported. Two minutes is an
ultra-short recording, enough for heart rate and RMSSD but not for the frequency-domain
measures, which want five. And breathing paced at 11 s per cycle puts its heart-rate swing at
0.09 Hz, inside the conventional low-frequency band, so an LF/HF ratio here would measure the
protocol rather than autonomic balance; none is computed.

This is pulse-rate variability at the finger, not R-R variability from an ECG. The two agree
closely at rest; deep breathing changes the pulse transit time with each breath, which shifts
finger beats against R waves by a few milliseconds. For a camera, which also sees the pulse
and not the R wave, the finger is the like-for-like reference.

Only references that can carry HRV are summarised, and two things rule one out. One is a
clock that the positive control in section 3 doubts: a loss missed or put at the wrong sample
dates a stretch one period off, which adds 16.7 ms steps to the successive differences that
RMSSD is built from. The other is a thin record, under 90% of its intervals kept or under
90 s of them: its gaps sit where the waveform was lost or clipped, and clipping can follow
the breathing or the hand's movement rather than fall at random.

Each activity is also compared with the same subject's stationary recording, where that one
carries HRV too, so differences between people do not masquerade as differences between
activities.

In [25]:
rows = []
for (subject, activity), table in BEATS.items():
    rows.append({'subject': subject, 'activity': activity, **hrv_metrics(table),
                 'kept_share': float(table.kept.mean()), 'kept_s': float(table.ibi[table.kept].sum())})
hrv_table = pd.DataFrame(rows)

# A reference carries HRV when the positive control in section 3 trusts its clock and it keeps at
# least 90% of its intervals and 90 s of them.
doubted = set(zip(shaky.subject, shaky.activity))
hrv_table['clock_doubted'] = [key in doubted for key in zip(hrv_table.subject, hrv_table.activity)]
hrv_table['thin'] = (hrv_table.kept_share < 0.9) | (hrv_table.kept_s < 90)
hrv_table['carries_hrv'] = ~hrv_table.clock_doubted & ~hrv_table.thin
usable = hrv_table[hrv_table.carries_hrv]
stationary = usable[usable.activity == 'stationary'].set_index('subject')
usable = usable.assign(rmssd_vs_stationary=usable.rmssd_ms / usable.subject.map(stationary.rmssd_ms))


def quartiles(s):
    return f'{s.median():.1f} ({s.quantile(0.25):.1f}-{s.quantile(0.75):.1f})'


print(f'references that carry HRV: {len(usable)} of {len(hrv_table)}')
print('set aside, clock doubted by the positive control:',
      [f'{r.subject}/{r.activity}' for r in hrv_table[hrv_table.clock_doubted].itertuples()] or 'none')
print('set aside, too thin (under 90% of intervals kept, or under 90 s of kept intervals):',
      hrv_table[hrv_table.thin][['subject', 'activity', 'kept_share', 'kept_s']].round(2)
      .to_dict('records') or 'none')
print()
summary = (usable.groupby('activity')
           .agg(recordings=('hr_bpm', 'size'),
                hr_bpm=('hr_bpm', quartiles), rmssd_ms=('rmssd_ms', quartiles),
                sdnn_ms=('sdnn_ms', quartiles), pnn50=('pnn50', 'median'),
                rmssd_vs_stationary=('rmssd_vs_stationary', 'median'), kept=('kept_share', 'median'))
           .reindex(list(ACTIVITIES)))
print('per activity over the references that carry HRV, median (interquartile range) over subjects;')
print('the ratio is to the same subject\'s stationary RMSSD, where that recording carries HRV too:')
print(summary.round(3).to_string())
print()
print(f'heart rate across all {len(hrv_table)} recordings '
      f'{hrv_table.hr_bpm.min():.0f}-{hrv_table.hr_bpm.max():.0f} bpm | RMSSD across the '
      f'{len(usable)} that carry HRV {usable.rmssd_ms.min():.0f}-{usable.rmssd_ms.max():.0f} ms')

references that carry HRV: 58 of 70
set aside, clock doubted by the positive control: ['005/reading', '005/stationary', '005/watching', '006/reading', '006/stationary', '013/watching', '015/watching']
set aside, too thin (under 90% of intervals kept, or under 90 s of kept intervals): [{'subject': '004', 'activity': 'deep', 'kept_share': 0.87, 'kept_s': 102.13}, {'subject': '004', 'activity': 'watching', 'kept_share': 0.88, 'kept_s': 105.09}, {'subject': '005', 'activity': 'talking', 'kept_share': 0.86, 'kept_s': 102.93}, {'subject': '006', 'activity': 'stationary', 'kept_share': 0.5, 'kept_s': 59.9}, {'subject': '008', 'activity': 'watching', 'kept_share': 0.86, 'kept_s': 95.59}, {'subject': '013', 'activity': 'watching', 'kept_share': 0.87, 'kept_s': 105.86}, {'subject': '014', 'activity': 'watching', 'kept_share': 0.88, 'kept_s': 103.17}]

per activity over the references that carry HRV, median (interquartile range) over subjects;
the ratio is to the same subject's stationary RMSSD, 

## 6. Deep breathing as the positive control
Paced at 6 s in and 5 s out, deep breathing should drive the heart rate up on every
inspiration and down on every expiration - respiratory sinus arrhythmia, at its largest at
slow breathing rates. If the reference cannot show that swing, it cannot support finer HRV
claims; if it shows it plainly, these recordings become the first test any camera method has
to pass.

Two measures, both from the kept intervals. The heart-rate series is put on a 4 Hz grid and
detrended, and its spectrum gives the dominant frequency between 0.04 and 0.4 Hz and the share
of power within 0.02 Hz of the paced rate, 1/11 Hz. The swing is the median, over consecutive
11 s windows, of the highest minus the lowest heart rate in the window. Both are computed for
every activity, so each subject's deep breathing is set against their own stationary
recording.

In [26]:
def rsa_measures(table, fs=HR_GRID_HZ, breath_s=BREATH_PERIOD_S):
    """Dominant frequency, paced-band share and per-breath swing of the heart-rate series."""
    kept = table.kept.to_numpy()
    t = table.t.to_numpy()[kept]
    hr = 60.0 / table.ibi.to_numpy()[kept]
    grid = np.arange(t[0], t[-1], 1.0 / fs)
    series = np.interp(grid, t, hr)
    detrended = series - np.polyval(np.polyfit(grid, series, 1), grid)
    power = np.abs(np.fft.rfft(detrended * np.hanning(grid.size))) ** 2
    freq = np.fft.rfftfreq(grid.size, 1.0 / fs)
    band = (freq >= 0.04) & (freq <= 0.4)
    paced = band & (np.abs(freq - 1.0 / breath_s) <= 0.02)
    width = int(round(breath_s * fs))
    swings = [series[i:i + width].max() - series[i:i + width].min()
              for i in range(0, grid.size - width + 1, width)]
    return {'peak_hz': float(freq[band][np.argmax(power[band])]),
            'paced_share': float(power[paced].sum() / power[band].sum()),
            'swing_bpm': float(np.median(swings))}


rsa = pd.DataFrame([{'subject': s, 'activity': a, **rsa_measures(table)}
                    for (s, a), table in BEATS.items()])
swing_by_subject = rsa.pivot(index='subject', columns='activity', values='swing_bpm')
share_by_subject = rsa.pivot(index='subject', columns='activity', values='paced_share')
deep = (rsa[rsa.activity == 'deep'].set_index('subject')
        .assign(breaths_per_min=lambda d: 60.0 * d.peak_hz,
                stationary_share=share_by_subject['stationary'],
                stationary_swing=swing_by_subject['stationary'],
                swing_ratio=lambda d: d.swing_bpm / d.stationary_swing))
on_pace = (deep.peak_hz - 1.0 / BREATH_PERIOD_S).abs() <= 0.02

print('deep breathing, per subject:')
print(deep[['peak_hz', 'breaths_per_min', 'paced_share', 'stationary_share', 'swing_bpm',
            'stationary_swing', 'swing_ratio']].round(3).to_string())
print()
print(f'dominant heart-rate rhythm within 0.02 Hz of the paced {1 / BREATH_PERIOD_S:.3f} Hz: '
      f'{int(on_pace.sum())}/{len(deep)} subjects; off pace: {list(deep.index[~on_pace]) or "none"}')
print(f'swing larger in deep breathing than in the same subject\'s stationary recording: '
      f'{int((deep.swing_ratio > 1).sum())}/{len(deep)} | '
      f'median ratio {deep.swing_ratio.median():.2f}, '
      f'median swing {deep.swing_bpm.median():.1f} against {deep.stationary_swing.median():.1f} bpm')
print(f'share of heart-rate power at the paced rate higher than in the same subject\'s '
      f'stationary recording: '
      f'{int((deep.paced_share > deep.stationary_share).sum())}/{len(deep)} | median '
      f'{deep.paced_share.median():.2f} against {deep.stationary_share.median():.2f}')
print()
print('all activities, medians over subjects:')
print(rsa.groupby('activity')[['peak_hz', 'paced_share', 'swing_bpm']].median()
      .reindex(list(ACTIVITIES)).round(3).to_string())

deep breathing, per subject:
         peak_hz  breaths_per_min  paced_share  stationary_share  swing_bpm  stationary_swing  swing_ratio
subject                                                                                                   
003        0.087            5.249        0.542             0.143     24.617            14.676        1.677
004        0.084            5.010        0.513             0.694     18.034             7.037        2.563
005        0.091            5.466        0.757             0.234     16.525             7.421        2.227
006        0.093            5.558        0.570             0.267     16.343             6.825        2.394
007        0.101            6.050        0.810             0.340     20.032             9.090        2.204
008        0.101            6.038        0.551             0.322     11.087             8.016        1.383
009        0.093            5.558        0.609             0.228     19.972             7.924        2.520
010     

## 7. Findings and decision

**Layout (confirmed).** 14 subjects x 5 activities: all 70 recordings present, 254,352 raw
frames (310.5 GiB), 3,387-4,019 per recording. Every frame is exactly one 8-bit 1280x1024 PGM
by size, the frame count equals all three per-frame logs in every recording, and the same 19
names sit in every folder.

**Frames are dated by the camera clock.** The camera's counter shows no dropped frame in any
recording, and its own clock runs at 30.017-30.018 fps with every interval between 33.27 and
33.36 ms. A uniform grid at each recording's own rate is never more than 0.046 ms off, so a
store can keep frames as a uniform sequence with one frame rate - the measured one, since
30.000 would drift 72 ms over two minutes. The PC's arrival stamps are prompt in most
recordings (0.09 ms above the envelope in the median one) but not in four: 005/reading arrives
74 ms late in the median and more than 10 ms late in 88% of its frames, 006/reading and
005/watching up to 46 and 26 ms late, and 015/reading stalls for up to 2.4 s in its last
seconds, after its oximeter link had failed. The camera clock, mapped onto the PC's through the
lower envelope, is what dates a frame. Exposure was set per recording, 16-30 ms, at gain 0.

**The oximeter record.** `px1_full.mat` holds the waveform (one byte per sample, top bit set,
0-127 once removed), the PC's read time of each sample and the sample count, and nothing else:
no pulse rate, SpO2 or beat marker. The text log equals `pulseOxTime` in all 70, and
`px1_partial.mat` is a leading part of the full record in all 70 (shorter in 12, by up to 265
samples), so only the full record is used.

**Its clock, rebuilt.** The device samples at 60 Hz (-15 ppm in the median recording), and the
reader lost 2,803 samples: 20 per recording in the median, one at a time in 62 recordings, at
most 14 in a row. Dating samples by their index alone would end the median recording 0.32 s
early, and the stamps wander 122 ms peak to peak around a straight line. The rebuilt clock fits
all 70, with no step down in any, and leaves 5.3 ms of read delay in the median recording. In
the positive control, 15,082 losses injected at three times the dataset's own rate came back
with the count off by 65 in all and 0.28 misplacements per run beyond the two-sample guard. The
rule mishandled more than 2% of them in seven recordings: six with slow or erratic reads
(median read delay 10-78 ms) - 005/stationary, 005/reading, 005/watching, 006/stationary,
006/reading and 015/watching - and 013/watching. The slowness was the PC's, not a device's:
005/reading's frames arrive 74 ms late and its oximeter samples 78 ms late.

**Coverage.** The reference starts 0.05-0.24 s after the first frame. In three recordings the
oximeter link failed before the camera stopped, leaving the last 8.4 s (003/reading), 7.4 s
(015/reading) and 3.2 s (009/stationary) of frames without a reference; those frames are left
out of any evaluation.

**Reference beats.** The steepest point of the upstroke is the fiducial. The systolic peak
agrees with it in most recordings (RMSSD +0.6 ms in the median) but not in seven, where the
maximum moves between the systolic and diastolic waves and RMSSD rises by 15 to 216 ms
(008/watching: 51 against 267 ms). With the minimum spacing tied to each recording's own beat
period, 96.7% of intervals are kept in the median recording; 499 are dropped next to a lost or
clipped sample, 24 for departing from their neighbours and 11 for range. A beat spans 52 of the
127 levels in the median recording, and the waveform sits at zero in 0.47% of samples, 8.2% at
worst (004/deep).

**The sampling is not the limit.** On a simulated train at this dataset's amplitudes, the
steepest upstroke refined between samples is timed to 0.35-0.49 ms after 7-bit rounding, which
adds 0.02-0.04 ms to an RMSSD of 18 ms. Without the refinement the same sampling costs 4.8 ms
per beat and 3.5 ms of RMSSD at 18 ms, a 20% error; taking the 11.7 ms noise RMSSD as the
error, as the sampling-ceiling script's `predicted` column does, would overstate it threefold.
Sub-sample refinement is essential and enough: whatever limits HRV from this reference, it is
not 60 Hz or 7 bits.

**HRV in the reference.** Pulse-rate variability at the finger over two minutes, from the 58
references that carry it: heart rate 50-111 bpm across all 70, RMSSD 8-96 ms. At rest the
median subject shows an RMSSD of 40 ms (interquartile 23-55) and an SDNN of 42 ms. Deep
breathing raises them to 55 and 78 ms, 1.24 times the same subject's resting RMSSD; reading
raises RMSSD by 17%, watching and talking by 4-5%, and talking lifts SDNN to 65 ms. That is
more variability than the 20-27 ms resting RMSSD of the MCD-rPPG subjects behind the earlier
HRV verdict, which makes this the more forgiving test. For scale, a camera's RMSSD stays within
10% of a 40 ms reference only if its beat times carry less than about 7 ms of independent
error.

**Deep breathing, the positive control, passes.** In 13 of 14 subjects the heart rate's
dominant rhythm sits within 0.02 Hz of the paced 0.091 Hz, and in 13 of 14 its swing per 11 s
window is larger than in the same subject's resting recording: 17.3 against 9.0 bpm in the
median, 1.88 times. Subject 011 is the exception for the rhythm (8.6 per minute, with little
power at the paced rate) and 014 for the swing (11.9 against 12.2 bpm). The rhythm alone does
not single out deep breathing - reading, watching and talking also peak near 0.09 Hz in the
median subject - but the share of power at the paced rate does, 0.58 against 0.16-0.35. The
measured rates, 4.3-6.1 per minute among the 13 on pace, also make the protocol's 5.45 per
minute a target rather than a ground truth for breathing rate; at two minutes one spectral bin
is 0.5 per minute.

**Decision.**
- The reference supports heart rate in all 70 recordings, trimmed where the oximeter stopped
  early, and HRV in 58 (section 5). Out for HRV are the seven recordings where the control
  mishandled more than 2% of losses, since their rebuilt clocks can misdate a stretch by a
  period - 006/stationary among them, with a fifth of its samples lost and 60 s of usable
  intervals - and five more that keep under 90% of their intervals: 004/deep, 004/watching,
  005/talking, 008/watching and 014/watching. That leaves 12, 12, 8, 13 and 13 recordings
  for stationary, reading, watching, talking and deep breathing, and subject 005 with deep
  breathing alone. All 14 deep-breathing recordings stay in for heart rate and the breathing
  control.
- Beat times on both sides come from the steepest upstroke, refined between samples; the
  systolic peak is not used.
- Frames are dated by the camera clock at each recording's measured rate, and the reference by
  its rebuilt clock, both on the PC's clock. The camera-to-finger offset is still estimated per
  recording, with a search wide enough for the oximeter's own delay as well as the pulse's
  travel to the finger, and it should come out much the same in every recording.
- The roles stay evaluation ones. Which of them the dataset can carry now depends on the
  frames: their Bayer layout and clipping, the face and its motion per activity, and whether a
  camera pulse can be timed against this reference at all. The frame sections settle that.

# Part 2 - The frames

*Added 2026-10-23, after part 1.* Part 1 settled the clocks and the reference from the logs
and the oximeter files alone. Part 2 reads the 254,352 raw frames: what the sensor recorded,
whether the app's own face, crop and gating code can work with it, and whether a camera pulse
can be timed against the finger reference at all - which decides whether the dataset can serve
the HRV questions, and as what. No frame is displayed: every result below is a statistic over
pixels.

### New in part 2
- **The Bayer layout is read from the data** (section 9), not taken from a manual or from
  OpenCV's naming: the two green sites are the diagonal pair whose levels agree, red is the
  site that reads higher on the face than across the frame, and a synthetic round trip picks
  the OpenCV code that returns R, G and B in that order.
- **The app's own code on these frames** (sections 10-14): the landmarker, the held box, the
  framing, face and motion gates, the crop and the POS reading are imported from
  `app/live_vitals` rather than restated, so every box, verdict and reading is the one the app
  would produce from these frames.
- **One pass over every frame** (section 11), cached per recording in
  `D:\QualityPhys\phase 3 exploration\NB_P3_33\`: the crop's colour trace, the app's reading
  and its motion verdict. The cache is keyed on the held box, the Bayer code and the source of
  every app module the pass uses, so a change to any of them recomputes rather than reuses.
- **Camera beats from the reference's fiducial** (section 13): the steepest systolic rise,
  refined between frames. The camera-to-finger offset is anchored per subject on all five
  recordings at once: their heart rates differ, so a one-beat slip lands at a different offset
  in each, while the true offset is common to all.

### Stated before the frames are read
- The layout must come out the same in every recording; section 9 stops otherwise.
- One camera and one oximeter: apart from each person's pulse travel, the offset should be
  about the same in every recording. Each recording is aligned within ±0.25 s of its subject's
  anchor - under half the shortest beat interval, so an alignment cannot slip by a beat - and
  one whose best alignment sits at that limit is reported, not moved.
- The app's readings are reported whatever its gates decide; the verdicts are findings in
  their own right here.

## 8. Part 2 setup
The app's modules are imported from the repository, found the way NB_P3_28 finds it. Part 1's
names stay in use: `recordings`, `camera` and the camera-log readers of section 2,
`REFERENCE`, `BEATS` and `refine`, and the HRV set `usable` of section 5.

In [27]:
import hashlib
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import cv2

# Walk up from the notebook until the repo root is found, identified by the MediaPipe asset (the
# same test NB_P3_28 uses; cwd is the nested notebook folder).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The app's code is imported, not restated: every box, crop, gate and reading below is the one the
# app would produce from these frames, and it moves if the app moves.
from app.live_vitals.capture.motion import motion_verdict
from app.live_vitals.capture.session import (detection_positions, face_verdict, framing_verdict,
                                             stable_box)
from app.live_vitals.config import (CLIP_LEN, CROP_PADDING, MAX_BOX_SIDE_LOST, MIN_CONFIDENCE,
                                    N_DETECT_FRAMES, SPECTRAL_ROI_FRACTION, TARGET_SIZE,
                                    WINDOW_STRIDE)
from app.live_vitals.estimators.hr_spectral import HRSpectral
from app.live_vitals.preprocess.face_box import landmark_box, make_landmarker
from app.live_vitals.preprocess.frames import crop_resize
from app.live_vitals.signal.spectral import pos, rgb_trace

PGM_HEADER = b'P5\n1280 1024\n255\n'   # checked on every frame read, never assumed
SITES = ('00', '01', '10', '11')       # (row, column) of a pixel inside its 2x2 Bayer cell
CACHE_DIR = Path(r'D:\QualityPhys\phase 3 exploration\NB_P3_33')
# Every app module the pass's outputs depend on; the cache is valid for this exact source only.
PASS_MODULES = ('config.py', 'preprocess/frames.py', 'signal/spectral.py', 'signal/hr.py',
                'estimators/base.py', 'estimators/hr_spectral.py', 'capture/motion.py')
FRAME_WORKERS = 8          # frames read, demosaiced and cropped in parallel threads
MATCH_TOL_S = 0.15         # a camera beat matches a reference beat this close (check_hrv_from_video.py)
ANCHOR_SEARCH_S = 2.0      # a subject's anchor offset is searched this far either way
ALIGN_HALF_WIDTH_S = 0.25  # each recording is aligned within this of its subject's anchor

LANDMARKER = make_landmarker()
print('repository:', REPO_ROOT)
print(f'app contract: crop {TARGET_SIZE}x{TARGET_SIZE} from a box padded by {CROP_PADDING:g}, colour '
      f'trace from its central {SPECTRAL_ROI_FRACTION:.0%}, windows of {CLIP_LEN} frames every '
      f'{WINDOW_STRIDE}, confidence gate {MIN_CONFIDENCE:g}, framing gate {MAX_BOX_SIDE_LOST:.0%} '
      f'of the box side')
print('frame cache:', CACHE_DIR, '| exists:', CACHE_DIR.exists())

repository: D:\code\QualityPhys - CRVSE Project
app contract: crop 72x72 from a box padded by 0.6, colour trace from its central 35%, windows of 160 frames every 80, confidence gate 0.65, framing gate 2% of the box side
frame cache: D:\QualityPhys\phase 3 exploration\NB_P3_33 | exists: True


## 9. The raw frame: Bayer layout, levels and clipping
Each frame is the sensor's colour mosaic: one 8-bit value per pixel, behind a red, green or
blue filter repeating in a 2x2 cell. Which site holds which colour is not in the logs -
MR-NIRP Car's README gives `rggb` for another camera on the same IMX249 sensor, but a sensor
window or a flip moves the phase - and a wrong guess swaps colours without raising anything.
Two properties of a mosaic settle it from the frames:
- **The two green sites sit on one diagonal, behind the same filter,** so their levels agree
  closely across the whole range, at the 10th, 50th and 90th percentiles as well as on
  average, while red and blue agree only where the light happens to balance them. Their fine
  detail also correlates more closely; that is printed beside the levels, as support.
- **Skin reflects far more red than blue.** On the face, the red site reads higher relative to
  green than it does across the whole frame, and the blue site lower. Finding the face needs
  no layout: the mean of each 2x2 cell is a half-resolution grey image, and the app's
  landmarker runs on that.

The evidence comes from the app's 12 detection samples of every recording. The logs record no
gamma setting, so whether the values are linear in light is not known; POS divides each
channel by its own mean, which makes it indifferent to a gain and nearly so to a gamma curve.
Levels and clipping are read on the central half of the face box, where the skin is: a pixel
at 255 no longer follows the light, and with it the pulse.

In [28]:
LEVEL_QUANTILES = (10, 50, 90)


def frame_path(folder, i):
    return Path(folder) / FRAME_DIR / f'Frame{i:05d}.pgm'


def read_raw(path):
    """One frame's mosaic as a (1024, 1280) uint8 array; its header is checked, not assumed."""
    with open(path, 'rb') as f:
        head, body = f.read(len(PGM_HEADER)), f.read()
    if head != PGM_HEADER or len(body) != FRAME_W * FRAME_H:
        raise ValueError(f'{path}: not one 8-bit {FRAME_W}x{FRAME_H} PGM')
    return np.frombuffer(body, dtype=np.uint8).reshape(FRAME_H, FRAME_W)


def site_planes(raw):
    """The four sites of the 2x2 cell, each as a half-resolution plane."""
    return {s: raw[int(s[0])::2, int(s[1])::2] for s in SITES}


def detail(plane):
    """A plane minus its 9x9 local mean, so that a correlation follows detail, not lighting."""
    p = plane.astype(np.float32)
    return (p - cv2.blur(p, (9, 9))).ravel()


def skin_slices(box):
    """The central half of a relative face box, as slices of the half-resolution site planes."""
    x0, y0, x1, y1 = box
    cx, cy = (x0 + x1) * FRAME_W / 4, (y0 + y1) * FRAME_H / 4
    hw, hh = (x1 - x0) * FRAME_W / 8, (y1 - y0) * FRAME_H / 8
    return (slice(max(0, int(cy - hh)), max(0, int(cy + hh))),
            slice(max(0, int(cx - hw)), max(0, int(cx + hw))))


def layout_evidence(row):
    """Medians over a recording's detection samples: the two diagonals' detail correlations, and
    each site's level and clipping over the whole frame and over the skin of the face."""
    samples = []
    for i in detection_positions(int(row.frames)):
        planes = site_planes(read_raw(frame_path(row.folder, i)))
        grey = np.mean([p.astype(np.float32) for p in planes.values()], axis=0).astype(np.uint8)
        face = landmark_box(np.repeat(grey[:, :, None], 3, axis=2), LANDMARKER)
        out = {'main_r': np.corrcoef(detail(planes['00']), detail(planes['11']))[0, 1],
               'anti_r': np.corrcoef(detail(planes['01']), detail(planes['10']))[0, 1]}
        for s, p in planes.items():
            out[f'frame_{s}'] = float(p.mean())
            for q, v in zip(LEVEL_QUANTILES, np.percentile(p, LEVEL_QUANTILES)):
                out[f'frame_p{q}_{s}'] = float(v)
            out[f'frame_clip_{s}'] = float((p == 255).mean())
            if face is not None:
                skin = p[skin_slices(face)]
                out[f'face_{s}'] = float(skin.mean())
                out[f'face_clip_{s}'] = float((skin == 255).mean())
                out[f'face_dark_{s}'] = float((skin == 0).mean())
        samples.append(out)
    samples = pd.DataFrame(samples)
    faces_found = int(samples['face_00'].notna().sum()) if 'face_00' in samples else 0
    return {'subject': row.subject, 'activity': row.activity, 'faces': faces_found,
            **samples.median(numeric_only=True).to_dict()}


def level_gap(ev, a, b):
    """How far two sites' frame levels differ, relative to their level: the worst of the mean and
    the 10th, 50th and 90th percentiles, so that two different filters cannot pass by matching on
    average alone."""
    stats = ['frame_'] + [f'frame_p{q}_' for q in LEVEL_QUANTILES]
    return max(abs(ev[f'{x}{a}'] - ev[f'{x}{b}']) / max((ev[f'{x}{a}'] + ev[f'{x}{b}']) / 2, 1.0)
               for x in stats)


def layout_of(ev):
    """The pattern as the colours of sites 00, 01, 10 and 11: green on the diagonal whose levels
    agree, red at the other site that is relatively redder on the face than over the frame."""
    if ev['faces'] == 0:
        return 'no face', np.nan
    anti = level_gap(ev, '01', '10') < level_gap(ev, '00', '11')
    green = ('01', '10') if anti else ('00', '11')
    others = [s for s in SITES if s not in green]
    g_face = np.mean([ev[f'face_{s}'] for s in green])
    g_frame = np.mean([ev[f'frame_{s}'] for s in green])
    redness = {s: (ev[f'face_{s}'] / g_face) / (ev[f'frame_{s}'] / g_frame) for s in others}
    red = max(others, key=redness.get)
    colour = {**{s: 'G' for s in green}, **{s: 'B' for s in others}, red: 'R'}
    return ''.join(colour[s] for s in SITES), redness[red] / min(redness.values())


def opencv_code(pattern):
    """The OpenCV Bayer code that returns R, G and B for a pattern, found by a round trip through a
    synthetic mosaic rather than read from OpenCV's names, which describe the cell from another
    pixel."""
    level = {'R': 200, 'G': 100, 'B': 50}
    mosaic = np.zeros((16, 16), np.uint8)
    for s, colour in zip(SITES, pattern):
        mosaic[int(s[0])::2, int(s[1])::2] = level[colour]
    for name in ('COLOR_BayerBG2RGB', 'COLOR_BayerGB2RGB', 'COLOR_BayerRG2RGB', 'COLOR_BayerGR2RGB'):
        rgb = cv2.cvtColor(mosaic, getattr(cv2, name))[4:-4, 4:-4].reshape(-1, 3).mean(axis=0)
        if np.allclose(rgb, [200, 100, 50]):
            return name
    raise RuntimeError(f'no OpenCV code returns R, G and B for {pattern}')


evidence = pd.DataFrame([layout_evidence(r) for r in recordings.itertuples()])
decided = evidence.apply(layout_of, axis=1, result_type='expand')
evidence['pattern'], evidence['red_vs_blue'] = decided[0], decided[1]
patterns = evidence.pattern.value_counts()
if len(patterns) != 1 or patterns.index[0] == 'no face':
    raise RuntimeError(f'the layout differs between recordings: {patterns.to_dict()}')
PATTERN = patterns.index[0]
BAYER_CODE_NAME = opencv_code(PATTERN)
BAYER_CODE = getattr(cv2, BAYER_CODE_NAME)
green = [s for s, c in zip(SITES, PATTERN) if c == 'G']
other = [s for s in SITES if s not in green]
red_site, blue_site = SITES[PATTERN.index('R')], SITES[PATTERN.index('B')]
green_gap = evidence.apply(lambda e: level_gap(e, *green), axis=1)
other_gap = evidence.apply(lambda e: level_gap(e, *other), axis=1)
green_r = evidence.anti_r if green == ['01', '10'] else evidence.main_r
other_r = evidence.main_r if green == ['01', '10'] else evidence.anti_r

print(f'faces found on the grey image in all {N_DETECT_FRAMES} samples: '
      f'{int((evidence.faces == N_DETECT_FRAMES).sum())}/{len(evidence)} recordings, '
      f'fewest {evidence.faces.min()}')
print(f'green sites {green[0]} and {green[1]}: levels {100 * green_gap.median():.2f}% apart in the '
      f'median recording (the worst of mean and percentiles), {100 * green_gap.max():.2f}% at most, '
      f'against {100 * other_gap.median():.1f}% ({100 * other_gap.min():.1f}% at least) for sites '
      f'{other[0]} and {other[1]} | detail correlation {green_r.median():.2f} against '
      f'{other_r.median():.2f}, higher in {int((green_r > other_r).sum())}/{len(evidence)} recordings')
print(f'red site {red_site}: relatively redder on the face than site {blue_site} by a factor '
      f'{evidence.red_vs_blue.median():.2f} in the median recording, {evidence.red_vs_blue.min():.2f} '
      f'at least')
print(f'pattern {PATTERN} (sites 00, 01, 10, 11) in {patterns.iloc[0]}/{len(evidence)} recordings '
      f'-> OpenCV {BAYER_CODE_NAME}')
print()
levels = evidence[['subject', 'activity']].copy()
for c, s in {'R': red_site, 'G': green[0], 'B': blue_site}.items():
    levels[f'face_{c}'] = evidence[f'face_{s}']
    levels[f'clip_{c}'] = evidence[f'face_clip_{s}']
    levels[f'frame_clip_{c}'] = evidence[f'frame_clip_{s}']
levels['dark'] = evidence[[f'face_dark_{s}' for s in SITES]].max(axis=1)
levels['clip_any'] = levels[['clip_R', 'clip_G', 'clip_B']].max(axis=1)
levels['exposure_ms'] = (camera.set_index(['subject', 'activity'])
                         .loc[list(zip(levels.subject, levels.activity)), 'exposure_ms']
                         .round().to_numpy())
print(f'skin of the face, median recording: R {levels.face_R.median():.0f}, '
      f'G {levels.face_G.median():.0f}, B {levels.face_B.median():.0f} of 255 | '
      f'R/G {(levels.face_R / levels.face_G).median():.2f}, '
      f'B/G {(levels.face_B / levels.face_G).median():.2f} | green across recordings '
      f'{levels.face_G.min():.0f}-{levels.face_G.max():.0f}')
print(f'skin pixels at 255, median recording: R {100 * levels.clip_R.median():.2f}%, '
      f'G {100 * levels.clip_G.median():.2f}%, B {100 * levels.clip_B.median():.2f}% | at 0 in any '
      f'site: {100 * levels.dark.max():.2f}% at worst | the whole frame at 255 in green, median '
      f'recording: {100 * levels.frame_clip_G.median():.1f}%')
clipped = levels[levels.clip_any > 0.01].sort_values('clip_any', ascending=False)
print(f'recordings with more than 1% of skin pixels at 255 in some colour: {len(clipped)}')
if len(clipped):
    print(clipped[['subject', 'activity', 'exposure_ms', 'face_R', 'face_G', 'face_B', 'clip_R',
                   'clip_G', 'clip_B']].round(3).to_string(index=False))

faces found on the grey image in all 12 samples: 70/70 recordings, fewest 12
green sites 01 and 10: levels 0.12% apart in the median recording (the worst of mean and percentiles), 3.51% at most, against 35.0% (6.8% at least) for sites 00 and 11 | detail correlation 0.83 against 0.69, higher in 69/70 recordings
red site 00: relatively redder on the face than site 11 by a factor 1.28 in the median recording, 1.04 at least
pattern RGGB (sites 00, 01, 10, 11) in 70/70 recordings -> OpenCV COLOR_BayerBG2RGB

skin of the face, median recording: R 123, G 114, B 72 of 255 | R/G 1.07, B/G 0.65 | green across recordings 65-173
skin pixels at 255, median recording: R 0.06%, G 0.05%, B 0.00% | at 0 in any site: 0.00% at worst | the whole frame at 255 in green, median recording: 3.0%
recordings with more than 1% of skin pixels at 255 in some colour: 11
subject   activity  exposure_ms  face_R  face_G  face_B  clip_R  clip_G  clip_B
    014 stationary         20.0 179.035 172.583 110.200   0.073   0.

## 10. The face, through the app's own code
Before it reads anything, the app finds the face with MediaPipe's landmarker on 12 frames
spread over the capture, holds one box for the whole capture - the median landmark box, made
square and padded by `CROP_PADDING` - and judges the capture on it: whether the padded box
fits the frame (the framing gate), and whether the face was found in the samples, alone, and
inside the box (the face checks). The same functions run here on every recording's 12 samples,
demosaiced with the code section 9 found.

The landmarker also runs on a gamma-encoded copy of the same samples (exponent 1/2.2), the
encoding of the video the models were trained on: if it found faces there that it misses in
the plain frames, a store would have to encode before detecting.

In [29]:
GAMMA_LUT = np.round(255.0 * (np.arange(256) / 255.0) ** (1 / 2.2)).astype(np.uint8)


def demosaic(raw):
    return cv2.cvtColor(raw, BAYER_CODE)


def face_audit(row):
    """The app's detection, held box, framing gate and face checks on one recording's samples."""
    samples = [demosaic(read_raw(frame_path(row.folder, i)))
               for i in detection_positions(int(row.frames))]
    box, found, faces = stable_box(samples, LANDMARKER)
    out = {'subject': row.subject, 'activity': row.activity, 'found': found, 'of': len(samples),
           'found_gamma': sum(landmark_box(GAMMA_LUT[s], LANDMARKER) is not None for s in samples)}
    if box is None:
        return out, None
    framing, report, _ = framing_verdict(box, FRAME_W, FRAME_H)
    checks, _, summary = face_verdict(faces, box)
    subject = np.array([f[0] for f in faces if f])
    out.update({'face_h': float(np.median(subject[:, 3] - subject[:, 1])),
                'face_w': float(np.median(subject[:, 2] - subject[:, 0])),
                'side_lost': report['frac_side_lost'], 'framing': framing, 'face_checks': checks,
                'second_face': summary['second_face_samples'],
                'inside_min': summary['face_inside_min']})
    return out, box


audits = [face_audit(r) for r in recordings.itertuples()]
face_table = pd.DataFrame([a for a, _ in audits])
BOXES = {(a['subject'], a['activity']): box for a, box in audits if box is not None}

print(f'face found in all {N_DETECT_FRAMES} samples: {int((face_table.found == N_DETECT_FRAMES).sum())}'
      f'/{len(face_table)} recordings, in the gamma-encoded copy '
      f'{int((face_table.found_gamma == N_DETECT_FRAMES).sum())}/{len(face_table)} | samples with a '
      f'face: plain {face_table.found.sum()}, gamma-encoded {face_table.found_gamma.sum()}, of '
      f'{face_table["of"].sum()}')
print(f'the face (landmark box) spans {100 * face_table.face_h.median():.0f}% of the frame\'s height '
      f'in the median recording ({100 * face_table.face_h.min():.0f}-'
      f'{100 * face_table.face_h.max():.0f}%) and {100 * face_table.face_w.median():.0f}% of its '
      f'width; padded by {CROP_PADDING:g}, the held box loses '
      f'{100 * face_table.side_lost.median():.0f}% of its side outside the frame in the median '
      f'recording ({100 * face_table.side_lost.min():.0f}-{100 * face_table.side_lost.max():.0f}%)')
print('framing gate:', face_table.framing.value_counts().to_dict(),
      '| face checks:', face_table.face_checks.value_counts().to_dict(),
      f'| a second face in {int((face_table.second_face > 0).sum())} recordings | least of the face '
      f'inside the held box at any sample: {100 * face_table.inside_min.min():.0f}%')
print()
print(face_table.groupby('activity')
      .agg(recordings=('found', 'size'),
           all_found=('found', lambda v: int((v == N_DETECT_FRAMES).sum())),
           face_h=('face_h', 'median'), side_lost=('side_lost', 'median'),
           framing_refused=('framing', lambda v: int((v == 'REJECT').sum())),
           checks_not_accepted=('face_checks', lambda v: int((v != 'ACCEPT').sum())))
      .reindex(list(ACTIVITIES)).round(3).to_string())
odd = face_table[(face_table.found < N_DETECT_FRAMES) | (face_table.face_checks != 'ACCEPT')]
if len(odd):
    print()
    print(odd[['subject', 'activity', 'found', 'found_gamma', 'face_checks', 'second_face',
               'inside_min']].round(3).to_string(index=False))

face found in all 12 samples: 70/70 recordings, in the gamma-encoded copy 70/70 | samples with a face: plain 840, gamma-encoded 840, of 840
the face (landmark box) spans 71% of the frame's height in the median recording (60-95%) and 49% of its width; padded by 0.6, the held box loses 18% of its side outside the frame in the median recording (6-34%)
framing gate: {'REJECT': 70} | face checks: {'ACCEPT': 70} | a second face in 0 recordings | least of the face inside the held box at any sample: 100%

            recordings  all_found  face_h  side_lost  framing_refused  checks_not_accepted
activity                                                                                  
stationary          14         14   0.715      0.185               14                    0
reading             14         14   0.731      0.194               14                    0
watching            14         14   0.709      0.201               14                    0
talking             14         14   0.729 

## 11. One pass over every frame
Every frame is demosaiced and cropped with its recording's held box to the app's 72x72 crop by
the app's own `crop_resize` - clamped to the frame where the padded box overflows it, which is
what the app would crop if its framing gate let the capture through. From one recording's
crops the pass keeps three things:
- the colour trace the app reads, the mean RGB of the central part of each crop (`rgb_trace`);
- the app's classical reading (`HRSpectral`: POS over the whole capture, read per window under
  the app's confidence and outlier gates), window by window;
- the app's motion verdict and its measure (`motion_verdict`).

Both readings take the frame rate from the camera's clock, and every frame is dated by that
clock mapped onto the PC's through section 2's lower envelope. The crops themselves are not
kept; the traces and readings, a few MB in all, are cached per recording, and a recording is
read again only when its held box, the Bayer code or the app's source differ from what its
cache was made with.

In [30]:
SOURCE_HASH = hashlib.sha256(b''.join((REPO_ROOT / 'app/live_vitals' / m).read_bytes()
                                      for m in PASS_MODULES)).hexdigest()[:16]


def frame_clock(row):
    """Frame times on the PC clock, from the camera's clock through section 2's lower envelope,
    and the frame rate on the camera's clock."""
    host, clock, _ = read_camera_logs(row.folder)
    a, b = lower_line(clock, host - clock)
    return a + (1.0 + b) * clock, (len(clock) - 1) / float(clock[-1])


def crops_of(row, box):
    """Every frame of a recording as the app's crop, in order. Frames are read, demosaiced and
    cropped in parallel threads: file reads and OpenCV's calls release the interpreter's lock."""
    def crop(i):
        return crop_resize(demosaic(read_raw(frame_path(row.folder, i))), box)
    crops = np.empty((int(row.frames), TARGET_SIZE, TARGET_SIZE, 3), dtype=np.uint8)
    with ThreadPoolExecutor(FRAME_WORKERS) as pool:
        for i, c in enumerate(pool.map(crop, range(int(row.frames)))):
            crops[i] = c
    return crops


def plain(value):
    """A JSON-ready copy of the app's results: numpy scalars and arrays become Python values."""
    if isinstance(value, dict):
        return {k: plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [plain(v) for v in value]
    return value.item() if isinstance(value, np.generic) else value


def frame_pass(row, box, fps):
    """The crops of one recording, reduced to the app's colour trace, reading and motion verdict."""
    crops = crops_of(row, box)
    reading = HRSpectral().estimate(crops, fps)
    motion, note, summary = motion_verdict(crops, fps)
    return rgb_trace(crops), {'hr_bpm': reading.value, 'status': reading.status,
                              'confidence': reading.confidence, 'detail': plain(reading.detail),
                              'motion': motion, 'motion_note': note, 'motion_summary': plain(summary)}


def cached_pass(row, box, fps):
    """One recording's pass, from the cache when the cache was made from the same inputs."""
    signature = {'box': [round(float(v), 6) for v in box], 'bayer': BAYER_CODE_NAME,
                 'frames': int(row.frames), 'fps': round(fps, 6), 'source': SOURCE_HASH}
    stem = CACHE_DIR / f'{row.subject}_{row.activity}'
    meta, traces = stem.with_suffix('.json'), stem.with_suffix('.npz')
    if meta.exists() and traces.exists():
        stored = json.loads(meta.read_text())
        if stored['signature'] == signature:
            with np.load(traces) as z:
                return z['trace'], stored['result'], False
    trace, result = frame_pass(row, box, fps)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(traces, trace=trace)
    meta.write_text(json.dumps({'signature': signature, 'result': result}))  # last: marks it complete
    return trace, result, True


FRAME_TIMES, TRACES, PASS = {}, {}, {}
t_start, read_now = time.time(), 0
for k, r in enumerate(recordings.itertuples(), 1):
    key = (r.subject, r.activity)
    FRAME_TIMES[key], fps = frame_clock(r)
    if key not in BOXES:
        continue
    print(f'\rframe pass {k}/{len(recordings)}: {r.subject}/{r.activity}  ', end='', flush=True)
    TRACES[key], PASS[key], fresh = cached_pass(r, BOXES[key], fps)
    PASS[key]['fps'] = fps
    read_now += fresh
print(f'\rframe pass: {len(PASS)} of {len(recordings)} recordings, {read_now} read from the frames and '
      f'{len(PASS) - read_now} from the cache, in {(time.time() - t_start) / 60:.1f} min')
print(f'frames per trace equal to the recording\'s frames: '
      f'{sum(len(TRACES[k]) == len(FRAME_TIMES[k]) for k in TRACES)}/{len(TRACES)} | '
      f'frame rate on the camera clock {min(p["fps"] for p in PASS.values()):.4f}-'
      f'{max(p["fps"] for p in PASS.values()):.4f} fps')

frame pass: 70 of 70 recordings, 0 read from the frames and 70 from the cache, in 0.0 min
frames per trace equal to the recording's frames: 70/70 | frame rate on the camera clock 30.0167-30.0180 fps


## 12. How much the head moves
The app measures motion on the crop itself: the frame-to-frame shift of its content,
band-passed to the cardiac band and expressed as an rms displacement in crop pixels per
analysis window. It refuses a capture whose median window moves 1.7 px or more, or 0.9 px with
a clear rhythm in half of its windows - thresholds set on still recordings only
(`calibrate_motion_gate.py`). Where the padded box is clamped to the frame (section 10), the
face fills more of the crop than in the recordings those thresholds came from, so the same
head movement reads as more crop pixels.

In [31]:
motion = pd.DataFrame([{'subject': s, 'activity': a, 'verdict': p['motion'],
                        **(p['motion_summary'] or {})} for (s, a), p in PASS.items()])
print('motion gate:', motion.verdict.value_counts().to_dict())
print(motion.groupby('activity')
      .agg(recordings=('verdict', 'size'), amp_px=('amp_median', 'median'),
           amp_px_worst=('amp_median', 'max'), clear_share=('clear_frac', 'median'),
           refused=('verdict', lambda v: int((v == 'REJECT').sum())))
      .reindex(list(ACTIVITIES)).round(3).to_string())
refused = motion[motion.verdict == 'REJECT'].sort_values('amp_median', ascending=False)
if len(refused):
    print()
    print('refused for motion:')
    print(refused[['subject', 'activity', 'amp_median', 'amp_p90', 'clear_frac',
                   'rhythm_when_clear']].round(3).to_string(index=False))

motion gate: {'ACCEPT': 70}
            recordings  amp_px  amp_px_worst  clear_share  refused
activity                                                          
stationary          14   0.043         0.068        0.239        0
reading             14   0.062         0.181        0.193        0
watching            14   0.064         0.153        0.271        0
talking             14   0.160         0.312        0.091        0
deep                14   0.065         0.106        0.213        0


## 13. Camera beats and the camera-to-finger offset
POS follows the light the skin reflects, and that light falls as blood volume rises, so the
systolic rise is where the POS pulse falls fastest. The camera's beat is therefore the
steepest rise of the inverted POS pulse - the same event as the finger's steepest upstroke -
refined between frames by section 4's parabola, under part 1's spacing and slope rules. The
other polarity is aligned as well, as a check on that sign.

The offset, finger minus camera, holds the pulse's longer path to the finger, the oximeter's
own processing, and whatever part of each device's latency to the PC a lower envelope cannot
remove; the camera's stamp may also mark the start of its 16-30 ms exposure rather than the
middle. It is the offset at which the median distance from each reference beat to the nearest
camera beat is smallest. `align_offset` in `check_hrv_from_video.py` counts the beats matched
within 150 ms instead, but that count barely moves when an alignment slips by a whole beat,
since most beat-to-beat changes are well under 150 ms; the median distance grows several-fold.
Only reference beats that border a kept interval are used, and only those that every candidate
keeps inside the camera's span, so that all candidates are scored on the same beats.

A slip is still possible in one recording whose heart rate barely changes. Across a subject's
five recordings it is not, because their heart rates differ, so a slip would land at a
different offset in each while the true offset is common to all. Each subject's anchor is
therefore taken from all five recordings at once, searched ±2 s, with the best distance more
than 0.25 s away printed beside it; an anchor more than 0.25 s from the median of all anchors
would be a slip and is replaced by that median. Every recording is then aligned within ±0.25 s
of its subject's anchor, and the share of reference beats with a camera beat within 150 ms is
reported.

In [32]:
def camera_pulse(trace, fps):
    """The app's POS pulse, inverted so that the systolic rise reads as a rising edge."""
    return -pos(trace, fps)


def camera_beats(pulse, fps):
    """Frame positions of the steepest rises, refined between frames: part 1's fiducial, spacing
    and slope rule, with the spacing tied to the recording's dominant period."""
    if not np.any(pulse):
        return np.array([])
    slope = np.gradient(pulse)
    f, p = welch(pulse - pulse.mean(), fs=fps, nperseg=min(len(pulse), int(16 * fps)))
    band = (f >= HR_MIN_BPM / 60.0) & (f <= HR_MAX_BPM / 60.0)
    period = fps / f[band][np.argmax(p[band])]
    idx, _ = find_peaks(slope, distance=max(1, int(BEAT_SPACING * period)))
    if idx.size < 3:
        return np.array([])
    local = pd.Series(slope[idx]).rolling(9, center=True, min_periods=3).median().to_numpy()
    idx = idx[slope[idx] >= 0.4 * local]
    return np.array([refine(slope, int(i)) for i in idx])


def on_pc_clock(key, positions):
    """Fractional frame positions as times on the PC clock."""
    t = FRAME_TIMES[key]
    return np.interp(positions, np.arange(t.size), t)


def reference_beats(key):
    """PC times of the reference beats, whether each borders an interval part 1 kept, and the kept
    flags of the intervals themselves."""
    table, (_, slots, times) = BEATS[key], REFERENCE[key]
    grid_s = np.concatenate([[table.t.iloc[0] - table.ibi.iloc[0]], table.t.to_numpy()])
    kept = table.kept.to_numpy()
    borders = np.zeros(grid_s.size, dtype=bool)
    borders[1:] |= kept
    borders[:-1] |= kept
    return np.interp(grid_s * OX_FS, slots, times), borders, kept


def nearest_gap(cam, ref):
    """Signed distance from each reference beat to the nearest camera beat, camera minus reference."""
    j = np.clip(np.searchsorted(cam, ref), 1, cam.size - 1)
    left, right = cam[j - 1] - ref, cam[j] - ref
    return np.where(np.abs(left) <= np.abs(right), left, right)


def gap_grid(key, beats, centre, half_width, step=0.0025):
    """Candidate offsets (finger minus camera) and, for each, the distance from every reference
    beat to its nearest camera beat - over the reference beats that border a kept interval and that
    every candidate keeps inside the camera's span."""
    offsets = np.arange(centre - half_width, centre + half_width + step / 2, step)
    cam = beats[key]
    ref, borders, _ = REF_BEATS[key]
    if cam.size < 3:
        return offsets, np.empty((offsets.size, 0))
    ref = ref[borders & (ref >= cam[0] + offsets[-1] + MATCH_TOL_S)
              & (ref <= cam[-1] + offsets[0] - MATCH_TOL_S)]
    return offsets, np.abs(np.array([nearest_gap(cam + o, ref) for o in offsets]))


def alignment_of(offsets, gaps):
    """The offset with the smallest median distance; its median distance, matched share and the
    best median distance more than ALIGN_HALF_WIDTH_S away from it."""
    if gaps.shape[1] == 0:
        return np.nan, np.nan, np.nan, np.nan
    median = np.median(gaps, axis=1)
    i = int(np.argmin(median))
    away = np.abs(offsets - offsets[i]) > ALIGN_HALF_WIDTH_S
    return (offsets[i], median[i], float(np.mean(gaps[i] <= MATCH_TOL_S)),
            median[away].min() if away.any() else np.nan)


CAM_BEATS, FLIPPED, REF_BEATS = {}, {}, {}
for key, trace in TRACES.items():
    pulse = camera_pulse(trace, PASS[key]['fps'])
    CAM_BEATS[key] = on_pc_clock(key, camera_beats(pulse, PASS[key]['fps']))
    FLIPPED[key] = on_pc_clock(key, camera_beats(-pulse, PASS[key]['fps']))
    REF_BEATS[key] = reference_beats(key)


def pooled(keys, beats):
    """One subject's recordings searched together: every recording's distances at each offset."""
    grids = [gap_grid(k, beats, 0.0, ANCHOR_SEARCH_S) for k in keys]
    return grids[0][0], np.concatenate([g for _, g in grids], axis=1)


anchor_rows = []
for subject in sorted({s for s, _ in TRACES}):
    keys = [k for k in TRACES if k[0] == subject]
    best, gap, matched, next_gap = alignment_of(*pooled(keys, CAM_BEATS))
    _, flip_gap, flip_matched, _ = alignment_of(*pooled(keys, FLIPPED))
    anchor_rows.append({'subject': subject, 'recordings': len(keys), 'offset_ms': 1000 * best,
                        'median_gap_ms': 1000 * gap, 'next_best_gap_ms': 1000 * next_gap,
                        'matched': matched, 'flipped_gap_ms': 1000 * flip_gap,
                        'flipped_matched': flip_matched})
anchors = pd.DataFrame(anchor_rows).set_index('subject')
centre_ms = anchors.offset_ms.median()
anchors['slip'] = (anchors.offset_ms - centre_ms).abs() > 1000 * ALIGN_HALF_WIDTH_S
anchors['used_ms'] = anchors.offset_ms.where(~anchors.slip & anchors.offset_ms.notna(), centre_ms)

align_rows, OFFSETS = [], {}
for key in TRACES:
    centre = anchors.used_ms[key[0]] / 1000
    best, gap, matched, _ = alignment_of(*gap_grid(key, CAM_BEATS, centre, ALIGN_HALF_WIDTH_S))
    OFFSETS[key] = best
    align_rows.append({'subject': key[0], 'activity': key[1], 'offset_ms': 1000 * best,
                       'from_anchor_ms': 1000 * (best - centre), 'median_gap_ms': 1000 * gap,
                       'matched': matched, 'at_limit': abs(best - centre) > ALIGN_HALF_WIDTH_S - 0.003})
alignment = pd.DataFrame(align_rows)

print(f'camera beats found per reference beat, median recording: '
      f'{np.median([CAM_BEATS[k].size / REF_BEATS[k][0].size for k in TRACES]):.2f}')
print(f'anchors - each subject\'s recordings together, searched ±{ANCHOR_SEARCH_S:g} s:')
print(anchors.round(3).to_string())
print(f'anchor offset median {centre_ms:+.0f} ms, interquartile '
      f'{anchors.offset_ms.quantile(0.25):+.0f} to {anchors.offset_ms.quantile(0.75):+.0f}, range '
      f'{anchors.offset_ms.min():+.0f} to {anchors.offset_ms.max():+.0f} | slips replaced by the '
      f'median: {int(anchors.slip.sum())} | the best alignment 0.25 s away sits '
      f'{(anchors.next_best_gap_ms / anchors.median_gap_ms).min():.1f} times further at least | the '
      f'inverted pulse aligns more closely than the other polarity in '
      f'{int((anchors.median_gap_ms < anchors.flipped_gap_ms).sum())}/{len(anchors)} subjects')
print()
print(f'every recording, aligned within ±{ALIGN_HALF_WIDTH_S:g} s of its subject\'s anchor:')
print(alignment.groupby('activity')
      .agg(recordings=('matched', 'size'), median_gap_ms=('median_gap_ms', 'median'),
           median_gap_worst=('median_gap_ms', 'max'), matched=('matched', 'median'),
           matched_worst=('matched', 'min'),
           from_anchor_ms=('from_anchor_ms', lambda v: v.abs().median()),
           at_limit=('at_limit', 'sum'))
      .reindex(list(ACTIVITIES)).round(3).to_string())
limit = alignment[alignment.at_limit | alignment.offset_ms.isna()]
if len(limit):
    print()
    print('best alignment at the search limit, or none:')
    print(limit.round(3).to_string(index=False))

camera beats found per reference beat, median recording: 0.99
anchors - each subject's recordings together, searched ±2 s:
         recordings  offset_ms  median_gap_ms  next_best_gap_ms  matched  flipped_gap_ms  flipped_matched   slip  used_ms
subject                                                                                                                  
003               5       -7.5         14.183            61.769    0.978          16.653            0.955  False     -7.5
004               5        7.5         13.794            22.079    0.967          20.519            0.966  False      7.5
005               5       22.5         14.148            28.081    0.998          17.518            0.986  False     22.5
006               5      -35.0         21.949            40.184    0.958          21.430            0.955  False    -35.0
007               5      -37.5         13.823            52.850    0.986          17.167            0.934  False    -37.5
008               5    

## 14. Heart rate from the camera, as the app reads it
The app's classical reading of each recording, from the pass: POS over the whole capture, read
in every 160-frame window, gated per window by spectral confidence and a MAD rule, and
summarised by the median of the kept windows - or refused, when too few survive. Each window
is set against the finger's heart rate over the same seconds: the mean of the reference's kept
intervals that end inside the window once the recording's offset is applied, scored only where
at least three do. The framing gate would refuse every recording whose padded box overflows
the frame (section 10); these readings are what the app would report if it let them through.

In [33]:
def reference_hr(key, t0, t1, min_intervals=3):
    """Finger heart rate over [t0, t1] on the camera's time line: the mean of the kept intervals
    that end inside it once the recording's offset is applied."""
    ref, _, kept = REF_BEATS[key]
    offset = OFFSETS[key] if np.isfinite(OFFSETS[key]) else anchors.used_ms[key[0]] / 1000
    ends, ibi = ref[1:] - offset, np.diff(ref)
    inside = kept & (ends >= t0) & (ends <= t1)
    return 60.0 / ibi[inside].mean() if inside.sum() >= min_intervals else np.nan


window_rows, recording_rows = [], []
for key, p in PASS.items():
    t, d = FRAME_TIMES[key], p['detail']
    for start, hr, conf, kept in zip(d['window_start'], d['window_hr'], d['window_confidence'],
                                     d['window_kept']):
        window_rows.append({'subject': key[0], 'activity': key[1], 'start': start, 'hr': hr,
                            'confidence': conf, 'kept': kept,
                            'ref_hr': reference_hr(key, t[start], t[start + CLIP_LEN - 1])})
    recording_rows.append({'subject': key[0], 'activity': key[1], 'status': p['status'],
                           'hr': p['hr_bpm'], 'ref_hr': reference_hr(key, t[0], t[-1]),
                           'windows': d['n_total'], 'read': len(d['window_hr']),
                           'kept': int(np.sum(d['window_kept']))})
windows = pd.DataFrame(window_rows)
windows['error'] = windows.hr - windows.ref_hr
readings = pd.DataFrame(recording_rows)
readings['error'] = readings.hr - readings.ref_hr
scored = windows.dropna(subset=['ref_hr'])
kept_scored = scored[scored.kept]

print(f'windows: {readings.windows.sum()} attempted, {len(windows)} with a spectral peak, '
      f'{int(windows.kept.sum())} kept by the app\'s gates | scored against the finger: '
      f'{len(scored)}, of them kept {len(kept_scored)}')
summary = pd.DataFrame({
    'windows': readings.groupby('activity').windows.sum(),
    'kept_share': readings.groupby('activity').kept.sum() / readings.groupby('activity').windows.sum(),
    'mae_kept': kept_scored.groupby('activity').error.apply(lambda e: e.abs().mean()),
    'within5_kept': kept_scored.groupby('activity').error.apply(lambda e: (e.abs() <= 5).mean()),
    'within5_all': scored.groupby('activity').error.apply(lambda e: (e.abs() <= 5).mean()),
    'reported': readings.groupby('activity').hr.apply(lambda h: int(h.notna().sum())),
    'reading_err_median': readings.groupby('activity').error.apply(lambda e: e.abs().median()),
    'reading_err_worst': readings.groupby('activity').error.apply(lambda e: e.abs().max()),
}).reindex(list(ACTIVITIES))
print(summary.round(3).to_string())
print()
print('the app\'s status per recording:', readings.status.value_counts().to_dict())
off = readings[readings.error.abs() > 5].sort_values('error', key=np.abs, ascending=False)
print(f'recordings whose reading is more than 5 bpm off the finger: {len(off)}')
if len(off):
    print(off[['subject', 'activity', 'status', 'hr', 'ref_hr', 'error', 'kept',
               'windows']].round(2).to_string(index=False))

windows: 3081 attempted, 3081 with a spectral peak, 2707 kept by the app's gates | scored against the finger: 3042, of them kept 2672
            windows  kept_share  mae_kept  within5_kept  within5_all  reported  reading_err_median  reading_err_worst
activity                                                                                                             
stationary      613       0.948     1.211         0.996        0.993        14               0.281              1.186
reading         619       0.963     1.337         0.980        0.972        14               0.383              1.145
watching        610       0.911     1.248         0.993        0.965        14               0.462              3.568
talking         619       0.658     2.039         0.948        0.848        13               1.056              2.448
deep            620       0.915     2.200         0.903        0.882        14               1.080              2.930

the app's status per recording: {'ok': 

## 15. Beat timing and HRV against the reference
Heart rate needs one frequency to be right; HRV needs every beat in its place. After
alignment, each reference beat that borders a kept interval is paired with the nearest camera
beat within 150 ms. Per recording this gives the share of those beats matched; the timing
error, the spread of camera minus reference; and RMSSD from the camera's intervals against the
reference's on the same beats - intervals whose two beats are both matched and kept, and
successive differences where three such beats follow one another. Section 4's quadrature rule
says what to expect: independent timing noise of SD σ adds √6·σ to RMSSD in quadrature, so a
40 ms RMSSD stays within 10% only for σ under about 7 ms. Only the 58 recordings that carry
HRV (section 5) are scored.

In [34]:
def beat_agreement(key):
    """Matched share, timing spread and RMSSD on shared beats, camera against reference."""
    cam = CAM_BEATS[key] + OFFSETS[key]
    ref, borders, kept = REF_BEATS[key]
    inside = borders & (ref >= cam[0] - MATCH_TOL_S) & (ref <= cam[-1] + MATCH_TOL_S)
    gap = nearest_gap(cam, ref)
    hit = inside & (np.abs(gap) <= MATCH_TOL_S)
    paired = kept & hit[:-1] & hit[1:]
    successive = paired[:-1] & paired[1:]
    ref_diff = np.diff(np.diff(ref))[successive]
    cam_diff = np.diff(np.diff(ref + gap))[successive]
    rmssd_ref = 1000 * np.sqrt(np.mean(ref_diff ** 2)) if successive.any() else np.nan
    rmssd_cam = 1000 * np.sqrt(np.mean(cam_diff ** 2)) if successive.any() else np.nan
    return {'subject': key[0], 'activity': key[1], 'matched': hit.sum() / max(inside.sum(), 1),
            'timing_sd_ms': 1000 * gap[hit].std() if hit.sum() > 2 else np.nan,
            'differences': int(successive.sum()), 'rmssd_ref_ms': rmssd_ref,
            'rmssd_cam_ms': rmssd_cam, 'rmssd_error': rmssd_cam / rmssd_ref - 1.0}


hrv_keys = set(zip(usable.subject, usable.activity))
timing = pd.DataFrame([beat_agreement(k) for k in TRACES
                       if k in hrv_keys and np.isfinite(OFFSETS[k]) and CAM_BEATS[k].size > 2])
# Independent jitter of SD s inflates RMSSD to sqrt(r**2 + 6 s**2): the jitter that would explain
# the camera's excess, to set beside the timing spread measured directly.
excess = np.clip(timing.rmssd_cam_ms ** 2 - timing.rmssd_ref_ms ** 2, 0, None)
timing['implied_sd_ms'] = np.sqrt(excess / 6)

print(f'recordings scored: {len(timing)} of the {len(hrv_keys)} that carry HRV')
print(timing.groupby('activity')
      .agg(recordings=('matched', 'size'), matched=('matched', 'median'),
           matched_worst=('matched', 'min'), timing_sd_ms=('timing_sd_ms', 'median'),
           implied_sd_ms=('implied_sd_ms', 'median'), rmssd_ref_ms=('rmssd_ref_ms', 'median'),
           rmssd_cam_ms=('rmssd_cam_ms', 'median'), rmssd_error=('rmssd_error', 'median'),
           within_10pct=('rmssd_error', lambda e: int((e.abs() <= 0.10).sum())))
      .reindex(list(ACTIVITIES)).round(3).to_string())
print()
print(f'timing spread across all scored recordings: median {timing.timing_sd_ms.median():.1f} ms, '
      f'best {timing.timing_sd_ms.min():.1f} ms | RMSSD within 10% of the finger in '
      f'{int((timing.rmssd_error.abs() <= 0.10).sum())}/{len(timing)} recordings, median error '
      f'{100 * timing.rmssd_error.median():+.0f}%')
best = timing.nsmallest(5, 'timing_sd_ms')
print('the five recordings timed most closely:')
print(best[['subject', 'activity', 'matched', 'timing_sd_ms', 'rmssd_ref_ms', 'rmssd_cam_ms',
            'rmssd_error']].round(3).to_string(index=False))

recordings scored: 58 of the 58 that carry HRV
            recordings  matched  matched_worst  timing_sd_ms  implied_sd_ms  rmssd_ref_ms  rmssd_cam_ms  rmssd_error  within_10pct
activity                                                                                                                          
stationary          12    1.000          0.921        17.415         13.707        39.251        59.108        0.327             2
reading             12    0.994          0.965        21.955         21.144        48.696        66.862        0.487             2
watching             8    0.993          0.904        22.143         20.280        41.302        70.234        0.469             0
talking             13    0.912          0.641        42.631         43.965        40.056       114.108        1.483             0
deep                13    1.000          0.929        22.060         15.557        54.937        78.043        0.339             2

timing spread across all scored rec

## 16. Findings and decision (part 2)

**The frame.** The layout is RGGB in all 70 recordings, read from the data: the two green
sites' levels are 0.12% apart in the median recording (3.51% at most, the worst of mean and
percentiles) against 35.0% (6.8% at least) for the other pair, their detail correlates more
closely in 69 of 70, and the red site is relatively redder on the face by a factor 1.28 in the
median recording, 1.04 at least. OpenCV's code for it is `COLOR_BayerBG2RGB`. The skin sits
mid-range (R 123, G 114, B 72 of 255 in the median recording, green 65-173 across recordings)
and almost none of it is clipped: 0.06% of skin pixels at 255 in red in the median recording.
Eleven recordings clip more than 1% in some colour, among them all five of subject 014, the
brightest face (green 156-173, 3-7% of its skin at 255 in red); it does not show in the pulse,
since 014 includes the recording timed most closely.

**The face and the app's gates.** The landmarker finds the face in all 840 samples, in the
plain frames and in the gamma-encoded copy alike, never sees a second face, and the face stays
wholly inside the held box at every sample. The faces are large: the landmark box spans 71% of
the frame's height in the median recording, 60-95% in all. Padded by 0.6, the app's square box
overflows the frame by 18% of its side in the median recording (6-34%), and the framing gate
refuses all 70. The motion gate accepts all 70: the crop moves 0.04 px in the median
stationary recording and 0.16 px in the median talking one, 0.31 px at worst, against
thresholds of 0.9 and 1.7 px.

**Heart rate.** Read through the app's own crop and POS reading, as if the framing gate had
let the recordings through, 69 of 70 get a reading - 67 ok, 2 degraded, one talking recording
refused - and none is more than 3.6 bpm off the finger: the median recording is 0.3 bpm off at
rest and about 1 bpm while talking or breathing deeply. The app's gates keep 88% of the
windows, 95% at rest and 66% while talking. Kept windows are 1.2-1.3 bpm off on average at
rest, reading and watching, 2.0 bpm talking and 2.2 bpm in deep breathing, where the rate
moves within a 5.3 s window; 90-99.6% of them are within 5 bpm.

**The offset.** The inverted POS pulse - the systolic rise - aligns more closely than the
other polarity in 12 of 14 subjects; in 006 the other polarity is closer by half a
millisecond, in 015 by 8 ms (16 against 24). Every anchor is unambiguous - the best alignment
0.25 s away sits at least 1.5 times further - and none needed replacing. The finger's beat is
stamped 33 ms before the camera's in the median subject (interquartile -38 to -11 ms, -68 to
+22 in all), although the pulse reaches the finger later than the face: the constant part of
each device's latency to the PC survives the lower envelopes, so the offset mixes the two
devices' latencies with the pulse's travel and cannot be read as a transit time. Within a
subject it holds across activities: each recording's own best sits a median 2.5-10 ms from its
subject's anchor, never at the ±0.25 s limit.

**Beat timing and HRV.** The camera finds one beat per reference beat (0.99 in the median
recording), and on the 58 recordings that carry HRV it matches 99-100% of the reference's
within 150 ms, except while talking (91%, 64% at worst). The timing spread is 17 ms in the
median stationary recording, 22 ms reading, watching or breathing deeply, 43 ms talking, and
10.3 ms at best (014/deep) - two to three times the 7 ms that a 40 ms RMSSD tolerates. RMSSD
from the camera runs 33% high at rest, 34% in deep breathing, 47-49% reading or watching and
148% talking, +54% in the median recording, and within 10% of the finger in 6 of 58. The
excess is what independent jitter predicts: 14-21 ms implied at rest, reading, watching and in
deep breathing, 44 ms talking. In deep breathing the measured spread, 22 ms, exceeds the 16 ms
implied, because part of it is slow: the face-to-finger delay itself follows the breath. Four
of the five recordings timed most closely, at 10-14 ms, land within 2-7% of the finger's
RMSSD; the fifth, 015 at rest, has a reference RMSSD of 23 ms, which 11 ms of jitter inflates
by 38%. The limit is the camera signal, not its 30 fps.

**Decision.**
- The frames are RGGB and are demosaiced with `COLOR_BayerBG2RGB`. Detection needs no gamma
  encoding: the plain frames give the landmarker every face the encoded ones do.
- The app's framing gate refuses every recording, because a face filling 60-95% of the frame's
  height cannot fit a box padded by 0.6. An evaluation store has to settle its crop first: the
  app's box clamped to the frame, as here, which leaves the aspect off the training stretch; a
  smaller padding, off the contract; or the frame padded out to hold the box. It also has to
  settle the colour - raw values, with the skin at R/G 1.07 and B/G 0.65, against the
  white-balanced, gamma-encoded video the models learned from. POS averages the crop and
  divides each channel by its own mean, so neither matters much to it; to a model both do. The
  preprocessing notebook decides, from these numbers.
- Heart rate: all 70 recordings, trimmed where the oximeter stopped early, against the camera
  with each recording's offset from section 13. Through the app's own POS reading they agree
  within 3.6 bpm: easy at rest, informative while talking and in deep breathing.
- HRV: the 58 recordings of section 5, beat by beat, with the same offsets. POS sets the
  classical baseline here - 17 ms of timing spread at rest, 24 ms overall, RMSSD a third high
  at rest and half high overall - and any camera HRV method has to beat it on these
  recordings.
- Alignment: the median distance between beats, pooled over a subject's recordings, replaces
  counting the beats matched within 150 ms, which a one-beat slip barely changes.
  `align_offset` in `check_hrv_from_video.py` (review item 3) should change the same way; on
  Rice the subjects' offsets lie within -68 to +22 ms, so there the width of the search was
  never the problem.
- The roles stay evaluation ones, for heart rate and HRV both; the store comes next, its crop
  rule first.

# Part 3 - HRV from the camera alone: taking out the timing jitter

*Added 2026-10-24, after part 2.* Part 2 timed the camera's beats 17-43 ms from the finger's
and found RMSSD inflated by about what independent timing jitter predicts. An app has no
finger: whatever it corrects, it has to estimate from the capture itself. Part 3 measures SDNN
and RMSSD as an app would compute them, over its 60 s capture and over whole recordings, and
tries two corrections that need nothing but the capture - one that predicts the jitter from
quality numbers the capture already provides, and one that measures it by timing three skin
regions separately.

### New in part 3
- **HRV as an app would compute it** (section 18): camera beats under part 1's interval rules,
  with no reference to pick them, against the finger over the same seconds - over the first
  and the last 60 s of each recording and over whole recordings - with three quality numbers
  per span: the app's window confidence, the share of intervals the rules drop, and how
  closely the beats follow the recording's typical beat shape.
- **A predicted correction** (section 19): each span's jitter predicted from those numbers by
  a fit on the other subjects' recordings, and its share taken out of SDNN and RMSSD.
- **A second pass over the frames** (section 20), cached beside part 2's: the mean colour of
  the forehead and of each cheek in every frame.
- **A measured correction** (sections 21-22): timed separately, three regions disagree only by
  the errors they do not share, which a three-cornered hat separates region by region. Their
  beats are fused, weighted by those errors, and what the fused beats still hold of them is
  taken out.

### Stated before the frames are read again
- Independent jitter of SD s inflates SDNN² by 2s² and RMSSD² by 6s². Both corrections
  subtract an estimate of that and nothing else, and neither takes a value below half of the
  one it corrects.
- The predicted correction is scored leave-one-subject-out: no subject's jitter is predicted
  by a fit that saw its recordings. The gate of section 18 is exploratory, set on these same
  recordings, and reported beside the full set, not instead of it.
- The hat sees only the jitter the regions do not share. Whatever they do share - the pulse's
  travel to the face against its travel to the finger, the light, the motion of the whole
  head - stays in the fused beats. The gap between the jitter the finger implies for the fused
  beats and the hat's estimate measures it, and bounds what fusing regions of one face can
  take out.
- The hat is solved on the heartbeats all three regions found. A region that misses many
  leaves the hardest beats out of the estimate, so the share found by all three is printed
  beside it.

## 17. Part 3 setup
The spans, thresholds and regions are fixed here, before anything is scored. Part 1's and part
2's names stay in use: the interval rules (`intervals`), the beats and offsets of section 13
(`CAM_BEATS`, `REF_BEATS`, `OFFSETS`), the app's readings of section 11 (`PASS`) and the HRV
set `usable` of section 5. The regions are fractions of the landmark box's width and height;
section 20 checks from the data that they fall on skin.

In [35]:
from scipy.stats import spearmanr

HRV_SPAN_S = 60.0             # the app's capture: SDNN and RMSSD over the first and the last 60 s
MIN_SPAN_INTERVALS = 30       # a span is scored only where both sides keep at least this many intervals
BEAT_SHAPE_S = (0.25, 0.35)   # a beat's shape: the pulse this long before and after its fiducial
JITTER_SHIFT_MS = 5.0         # jitter is fitted as log(s + 5 ms): a span with none stays on the scale
CORRECTION_FLOOR = 0.5        # no correction takes SDNN or RMSSD below half of the value it corrects
GATE_CONFIDENCE = 0.80        # exploratory gate, set on these recordings: the app's median window
GATE_DROPPED = 0.02           # confidence at least this, and at most this share of intervals dropped
# Skin regions as fractions of the landmark box (x0, y0, x1, y1), left and right as in the image: the
# forehead between the top of the landmarks and the brows, each cheek between the eye and the nostril.
REGIONS = {'forehead': (0.30, 0.05, 0.70, 0.20),
           'cheek_left': (0.14, 0.50, 0.36, 0.68),
           'cheek_right': (0.64, 0.50, 0.86, 0.68)}
REGION_DIR = CACHE_DIR / 'regions'
MIN_REGION_CELLS = 1000       # a region needs this many 2x2 cells inside the frame to be read
JITTER_FLOOR_MS = 5.0         # no region counts as timed closer than this: half part 2's best
SPANS = ('first 60 s', 'last 60 s', 'whole')
HRV_KEYS = sorted(set(zip(usable.subject, usable.activity)) & set(TRACES))


def span_bounds(key):
    """The scored spans on the PC clock: the first and the last HRV_SPAN_S of the frames, and all of
    them."""
    t = FRAME_TIMES[key]
    return {'first 60 s': (t[0], t[0] + HRV_SPAN_S), 'last 60 s': (t[-1] - HRV_SPAN_S, t[-1]),
            'whole': (t[0], t[-1])}


def beat_intervals(beats):
    """Intervals between beats under part 1's rules - 35-180 bpm, within 25% of the local median -
    which need no reference: their end times, lengths and kept flags."""
    table = intervals(pd.DataFrame({'slope_s': beats, 'peak_s': beats, 'near_gap': False,
                                    'clipped': False}))
    return table.t.to_numpy(), table.ibi.to_numpy(), table.kept.to_numpy()


def finger_intervals(key):
    """The finger's intervals and part 1's kept flags, their ends moved onto the camera's time line
    by the recording's offset."""
    ref, _, kept = REF_BEATS[key]
    offset = OFFSETS[key] if np.isfinite(OFFSETS[key]) else anchors.used_ms[key[0]] / 1000
    return ref[1:] - offset, np.diff(ref), kept


def span_hrv(ends, ibi, kept, t0, t1):
    """SDNN and RMSSD (ms) over the kept intervals that end inside [t0, t1], and their number. RMSSD
    takes a successive difference only where both intervals are kept."""
    k = kept & (ends >= t0) & (ends <= t1)
    pair = k[1:] & k[:-1]
    sdnn = 1000 * np.std(ibi[k], ddof=1) if k.sum() > 2 else np.nan
    rmssd = 1000 * np.sqrt(np.mean(np.diff(ibi)[pair] ** 2)) if pair.sum() > 2 else np.nan
    return sdnn, rmssd, int(k.sum())


def corrected(value, removed):
    """sqrt(value² - removed), never below CORRECTION_FLOOR x value; `removed` in ms², the share of
    value² that a correction attributes to jitter."""
    return np.sqrt(np.maximum(value ** 2 - removed, (CORRECTION_FLOOR * value) ** 2))


def error_scores(value, finger):
    """Error against the finger (value / finger - 1): median, median absolute, the shares within 10%
    and 20%, and the worst."""
    e = (value / finger - 1).dropna()
    return pd.Series({'spans': len(e), 'median': e.median(), 'median_abs': e.abs().median(),
                      'within_10': (e.abs() <= 0.10).mean(), 'within_20': (e.abs() <= 0.20).mean(),
                      'worst': e.abs().max()})


def by_activity(df, value, finger, columns=('spans', 'median', 'within_10', 'within_20')):
    """error_scores per activity and over all of df."""
    rows = {a: error_scores(df.loc[df.activity == a, value], df.loc[df.activity == a, finger])
            for a in ACTIVITIES}
    rows['all'] = error_scores(df[value], df[finger])
    table = pd.DataFrame(rows).T[list(columns)]
    if 'spans' in table:
        table['spans'] = table.spans.astype(int)
    return table


print(f'recordings that carry HRV: {len(HRV_KEYS)} | spans: {", ".join(SPANS)}, each scored where '
      f'camera and finger keep at least {MIN_SPAN_INTERVALS} intervals')

recordings that carry HRV: 58 | spans: first 60 s, last 60 s, whole, each scored where camera and finger keep at least 30 intervals


## 18. HRV from the camera alone
An app that reports HRV has the camera's beats and nothing else: no reference picks the beats,
and the rules that drop an interval have to work from the camera's intervals alone. Section 15
scored the camera on the finger's beats, which isolates the timing. Here the camera's
intervals are kept or dropped by part 1's rules - 35-180 bpm, within 25% of the local median -
and their SDNN and RMSSD are set against the finger's, from its own kept intervals, over the
same seconds: each side's intervals that end inside the span once the recording's offset is
applied. The spans are the first and the last 60 s of each recording, the app's capture
length, and the whole recording; a span is scored where both sides keep at least 30 intervals.

Three numbers per span describe the capture's quality without the finger, as candidates for
predicting its jitter:
- the app's own window confidence (section 14), the median over the windows inside the span;
- the share of the camera's intervals that the rules drop;
- how closely the beats follow the recording's typical beat: each beat's correlation with the
  median beat shape, the pulse from 0.25 s before its fiducial to 0.35 s after, the median
  over the span.

The jitter each excess implies - the square root of (camera² - finger²) / 2 for SDNN, / 6 for
RMSSD - is computed per span; under independent jitter the two agree. The exploratory gate
keeps the spans whose median window confidence is at least 0.80 and that drop at most 2% of
the camera's intervals.

In [36]:
def beat_shapes(key):
    """Each camera beat's PC time and the correlation of its shape - the pulse from BEAT_SHAPE_S[0]
    before its fiducial to BEAT_SHAPE_S[1] after it - with the recording's median beat. A beat timed
    on noise rather than on the pulse's rise departs from that shape."""
    fps = PASS[key]['fps']
    pulse = camera_pulse(TRACES[key], fps)
    frames = camera_beats(pulse, fps)
    before, after = (int(round(s * fps)) for s in BEAT_SHAPE_S)
    at = np.round(frames).astype(int)
    whole = (at >= before) & (at + after < pulse.size)
    segments = np.array([pulse[i - before:i + after + 1] for i in at[whole]])
    segments -= segments.mean(axis=1, keepdims=True)
    typical = np.median(segments, axis=0)
    return on_pc_clock(key, frames[whole]), np.array([np.corrcoef(s, typical)[0, 1] for s in segments])


span_rows = []
for key in HRV_KEYS:
    c_end, c_ibi, c_kept = beat_intervals(CAM_BEATS[key])
    f_end, f_ibi, f_kept = finger_intervals(key)
    shape_t, shape_r = beat_shapes(key)
    t, detail = FRAME_TIMES[key], PASS[key]['detail']
    w_start = np.asarray(detail['window_start'], dtype=int)
    w_conf = np.asarray(detail['window_confidence'], dtype=float)
    w_first, w_last = t[w_start], t[np.minimum(w_start + CLIP_LEN - 1, t.size - 1)]
    for span, (t0, t1) in span_bounds(key).items():
        sdnn_c, rmssd_c, n_c = span_hrv(c_end, c_ibi, c_kept, t0, t1)
        sdnn_f, rmssd_f, n_f = span_hrv(f_end, f_ibi, f_kept, t0, t1)
        ending = (c_end >= t0) & (c_end <= t1)
        windows_in = (w_first >= t0) & (w_last <= t1)
        span_rows.append({
            'subject': key[0], 'activity': key[1], 'span': span, 'n_cam': n_c, 'n_finger': n_f,
            'sdnn_cam': sdnn_c, 'sdnn_finger': sdnn_f, 'rmssd_cam': rmssd_c, 'rmssd_finger': rmssd_f,
            'confidence': np.median(w_conf[windows_in]) if windows_in.any() else np.nan,
            'dropped': 1.0 - c_kept[ending].mean() if ending.any() else np.nan,
            'beat_shape': np.median(shape_r[(shape_t >= t0) & (shape_t <= t1)])})
span_table = pd.DataFrame(span_rows)
span_table['scored'] = ((span_table.n_cam >= MIN_SPAN_INTERVALS)
                        & (span_table.n_finger >= MIN_SPAN_INTERVALS))
span_table['capture'] = span_table.span != 'whole'
span_table['gated'] = ((span_table.confidence >= GATE_CONFIDENCE)
                       & (span_table.dropped <= GATE_DROPPED))
# Independent jitter of SD s adds 2s² to SDNN² and 6s² to RMSSD²: the jitter each excess implies.
for metric, factor in (('sdnn', 2), ('rmssd', 6)):
    excess = span_table[f'{metric}_cam'] ** 2 - span_table[f'{metric}_finger'] ** 2
    span_table[f'jitter_{metric}'] = np.sqrt(np.clip(excess, 0, None) / factor)

captures = span_table[span_table.scored & span_table.capture]
wholes = span_table[span_table.scored & ~span_table.capture]
print(f'spans scored: {len(captures)} of {2 * len(HRV_KEYS)} 60 s captures, {len(wholes)} of '
      f'{len(HRV_KEYS)} whole recordings | error = camera / finger - 1')
for metric in ('sdnn', 'rmssd'):
    medians = captures.groupby('activity')[[f'{metric}_finger', f'{metric}_cam']].median()
    table = by_activity(captures, f'{metric}_cam', f'{metric}_finger').join(
        medians.rename(columns=lambda c: c.split('_')[1] + '_ms'))
    table.loc['all', ['finger_ms', 'cam_ms']] = captures[[f'{metric}_finger',
                                                         f'{metric}_cam']].median().to_numpy()
    print()
    print(f'{metric.upper()} from the camera alone, 60 s captures:')
    print(table[['spans', 'finger_ms', 'cam_ms', 'median', 'within_10', 'within_20']].round(3)
          .to_string())
print()
print('whole recordings, all activities:')
print(pd.DataFrame({m.upper(): error_scores(wholes[f'{m}_cam'], wholes[f'{m}_finger'])
                    for m in ('sdnn', 'rmssd')}).T.astype({'spans': int}).round(3).to_string())
print()
r_both, _ = spearmanr(captures.jitter_sdnn, captures.jitter_rmssd)
by_act = captures.groupby('activity').jitter_rmssd.median().reindex(list(ACTIVITIES)).round(1)
print(f'jitter the excess implies, median 60 s capture: {captures.jitter_sdnn.median():.1f} ms from '
      f'SDNN, {captures.jitter_rmssd.median():.1f} ms from RMSSD, Spearman {r_both:+.2f} between the '
      f'two | from RMSSD by activity (ms): {by_act.to_dict()}')
print('Spearman of each quality number with that jitter: ' + ', '.join(
    f'{c} {spearmanr(captures[c], captures.jitter_rmssd)[0]:+.2f}'
    for c in ('confidence', 'dropped', 'beat_shape')))
gated = captures[captures.gated]
passed = gated.activity.value_counts().reindex(list(ACTIVITIES)).fillna(0).astype(int)
print(f'exploratory gate - median window confidence at least {GATE_CONFIDENCE:g}, at most '
      f'{GATE_DROPPED:.0%} of the intervals dropped: {len(gated)} of {len(captures)} captures pass, '
      f'by activity {passed.to_dict()}')
print(pd.DataFrame({m.upper(): error_scores(gated[f'{m}_cam'], gated[f'{m}_finger'])
                    for m in ('sdnn', 'rmssd')}).T.astype({'spans': int}).round(3).to_string())
print()
worst = captures.assign(sdnn_error=captures.sdnn_cam / captures.sdnn_finger - 1)
worst = worst.reindex(worst.sdnn_error.abs().sort_values(ascending=False).index).head(6)
print('largest SDNN errors among the 60 s captures:')
print(worst[['subject', 'activity', 'span', 'sdnn_finger', 'sdnn_cam', 'sdnn_error', 'confidence',
             'dropped', 'beat_shape']].round(3).to_string(index=False))

spans scored: 116 of 116 60 s captures, 58 of 58 whole recordings | error = camera / finger - 1

SDNN from the camera alone, 60 s captures:
            spans  finger_ms  cam_ms  median  within_10  within_20
stationary     24     42.675  52.966   0.082      0.542      0.625
reading        24     47.742  54.779   0.104      0.500      0.667
watching       16     47.159  70.796   0.179      0.188      0.562
talking        26     58.471  84.711   0.545      0.000      0.115
deep           26     70.890  78.501   0.042      0.692      0.885
all           116     53.808  64.512   0.162      0.397      0.569

RMSSD from the camera alone, 60 s captures:
            spans  finger_ms  cam_ms  median  within_10  within_20
stationary     24     39.290  59.724   0.363      0.083      0.375
reading        24     47.828  67.782   0.501      0.208      0.292
watching       16     42.036  74.025   0.661      0.125      0.125
talking        26     38.508  89.034   1.344      0.000      0.000
deep       

## 19. A predicted correction
If a span's jitter could be read off its quality numbers, it could be subtracted: SDNN² - 2s²
and RMSSD² - 6s². Three predictions of s are compared, each a least-squares fit of log(s + 5
ms), s the jitter the RMSSD excess implies, on the 60 s captures of the other thirteen
subjects: a constant (the other subjects' typical jitter), the window confidence alone, and
the three quality numbers together, the beat-shape correlation through its Fisher transform.
Every subject, its whole recordings included, is predicted by a fit that never saw it.

As a ceiling no capture alone can reach, SDNN is also corrected with the jitter its own RMSSD
excess implies. That needs the finger, but it shows how much of SDNN's error is jitter at all.

In [37]:
JITTER_MODELS = {'constant': [], 'confidence': ['confidence'],
                 'quality': ['confidence', 'dropped', 'shape_z']}
span_table['shape_z'] = np.arctanh(np.clip(span_table.beat_shape, -0.999, 0.999))


def fit_jitter(train, columns):
    """Least squares of log(s + JITTER_SHIFT_MS) on the quality numbers, s the jitter the RMSSD
    excess implies. Returns the coefficients and a predictor of s in ms, whose back-transform carries
    the mean of the exponentiated residuals (Duan's smearing): a mean jitter, not a median one."""
    def design(df):
        return np.column_stack([np.ones(len(df))] + [df[c].to_numpy(float) for c in columns])
    y = np.log(train.jitter_rmssd.to_numpy() + JITTER_SHIFT_MS)
    beta, *_ = np.linalg.lstsq(design(train), y, rcond=None)
    smear = np.mean(np.exp(y - design(train) @ beta))
    return beta, lambda df: np.clip(np.exp(design(df) @ beta) * smear - JITTER_SHIFT_MS, 0, None)


def loso_jitter(columns):
    """Every span's predicted jitter, each subject's from a fit on the other subjects' scored 60 s
    captures alone."""
    out = pd.Series(np.nan, index=span_table.index)
    train = span_table[span_table.scored & span_table.capture]
    for subject in span_table.subject.unique():
        _, predict = fit_jitter(train[train.subject != subject], columns)
        mine = span_table.subject == subject
        out[mine] = predict(span_table[mine])
    return out


floor_hits = {}
for name, columns in JITTER_MODELS.items():
    span_table[f'jitter_{name}'] = loso_jitter(columns)
    for metric, factor in (('sdnn', 2), ('rmssd', 6)):
        value, removed = span_table[f'{metric}_cam'], factor * span_table[f'jitter_{name}'] ** 2
        span_table[f'{metric}_{name}'] = corrected(value, removed)
        hit = value ** 2 - removed < (CORRECTION_FLOOR * value) ** 2
        floor_hits[f'{metric} {name}'] = int((hit & span_table.scored & span_table.capture).sum())
# The ceiling: SDNN corrected with the jitter its own RMSSD excess implies, which needs the finger.
span_table['sdnn_ceiling'] = corrected(span_table.sdnn_cam, 2 * span_table.jitter_rmssd ** 2)

captures = span_table[span_table.scored & span_table.capture]
wholes = span_table[span_table.scored & ~span_table.capture]
for label, df in (('60 s captures', captures), ('whole recordings', wholes),
                  ('60 s captures through the gate', captures[captures.gated])):
    print(f'{label} ({len(df)}): the camera, and corrected with the jitter each model predicts')
    for metric in ('sdnn', 'rmssd'):
        rows = {'camera': 'cam', 'constant': 'constant', 'confidence': 'confidence',
                'quality numbers': 'quality'}
        if metric == 'sdnn':
            rows['ceiling (finger)'] = 'ceiling'
        print(f'{metric.upper()}:')
        print(pd.DataFrame({r: error_scores(df[f'{metric}_{c}'], df[f'{metric}_finger'])
                            for r, c in rows.items()}).T.drop(columns='spans').round(3).to_string())
    print()

print('60 s captures by activity, the camera against the quality numbers\' correction:')
for metric in ('sdnn', 'rmssd'):
    columns = ('median', 'within_10', 'within_20')
    print(f'{metric.upper()}:')
    print(by_activity(captures, f'{metric}_cam', f'{metric}_finger', columns)
          .join(by_activity(captures, f'{metric}_quality', f'{metric}_finger', columns),
                lsuffix='_cam', rsuffix='_corrected').round(3).to_string())
print()
print('60 s captures where a correction hit the floor:', floor_hits)
r_pred, _ = spearmanr(captures.jitter_quality, captures.jitter_rmssd)
print(f'predicted against implied jitter, 60 s captures: Spearman {r_pred:+.2f}, median '
      f'{captures.jitter_quality.median():.1f} ms predicted and '
      f'{captures.jitter_rmssd.median():.1f} ms implied')
beta, _ = fit_jitter(captures, JITTER_MODELS['quality'])
print(f'quality model on all subjects: log(s + {JITTER_SHIFT_MS:g} ms) = {beta[0]:+.2f} '
      f'{beta[1]:+.2f} confidence {beta[2]:+.2f} dropped {beta[3]:+.2f} atanh(shape)')

60 s captures (116): the camera, and corrected with the jitter each model predicts
SDNN:
                  median  median_abs  within_10  within_20  worst
camera             0.162       0.162      0.397      0.569  4.209
constant          -0.019       0.162      0.345      0.586  3.561
confidence         0.018       0.107      0.440      0.759  2.911
quality numbers    0.004       0.100      0.509      0.750  2.469
ceiling (finger)   0.008       0.042      0.750      0.845  2.363
RMSSD:
                 median  median_abs  within_10  within_20  worst
camera            0.631       0.631      0.112      0.250  7.818
constant         -0.056       0.351      0.129      0.293  6.406
confidence        0.122       0.293      0.138      0.397  5.320
quality numbers   0.117       0.295      0.147      0.379  4.579

whole recordings (58): the camera, and corrected with the jitter each model predicts
SDNN:
                  median  median_abs  within_10  within_20  worst
camera             0.135 

## 20. Three skin regions, read from the frames again
The three-cornered hat needs the same heartbeats timed by three sources whose errors are
independent. Three patches of skin on one face share the pulse - the same heart, nearly the
same path - but not their pixels: the forehead, the left cheek and the right cheek, placed on
the landmark box as fractions of its width and height (section 17) and held for the whole
recording, as the app holds its box. Their mean R, G and B come straight from the mosaic, each
colour averaged over its own sites; demosaicing interpolates each colour from those same
sites, so over the thousands of cells in a region the means agree, and no interpolation
reaches across a region's edge.

This second pass reads the frames of the 58 recordings that carry HRV and caches each one's
three colour traces in a `regions` folder beside part 2's cache, keyed on the regions'
rectangles, the layout and the frame count. Placement is checked from the data, as no frame is
displayed: a region on skin keeps the skin's colour balance, so its R/G and B/G sit near those
of the app's crop, while hair, a brow or the background would pull them away.

In [38]:
COLOUR_SITES = {'R': [red_site], 'G': list(green), 'B': [blue_site]}
face_sizes = face_table.set_index(['subject', 'activity'])[['face_w', 'face_h']]


def region_rectangles(key):
    """Each region's rows and columns on the mosaic, on whole 2x2 cells, or None where fewer than
    MIN_REGION_CELLS cells fall inside the frame. The regions sit on the landmark box: centred where
    the held box is centred, with the median landmark width and height of the recording's samples."""
    x0, y0, x1, y1 = BOXES[key]
    width, height = face_sizes.loc[key]
    left, top = (x0 + x1) / 2 - width / 2, (y0 + y1) / 2 - height / 2
    out = {}
    for name, (a, b, c, d) in REGIONS.items():
        c0, c1 = (int(np.clip((left + f * width) * FRAME_W, 0, FRAME_W)) // 2 * 2 for f in (a, c))
        r0, r1 = (int(np.clip((top + f * height) * FRAME_H, 0, FRAME_H)) // 2 * 2 for f in (b, d))
        cells = max(0, r1 - r0) * max(0, c1 - c0) // 4
        out[name] = (slice(r0, r1), slice(c0, c1)) if cells >= MIN_REGION_CELLS else None
    return out


def region_colours(raw, rectangles):
    """Mean R, G and B of each region straight from the mosaic, each colour over its own sites. A
    demosaiced frame interpolates each colour from the same sites, so over thousands of cells the
    means agree, without the interpolation's edges."""
    out = np.full((len(rectangles), 3), np.nan)
    for k, place in enumerate(rectangles.values()):
        if place is None:
            continue
        patch = raw[place]
        for c, sites in enumerate(COLOUR_SITES.values()):
            out[k, c] = np.mean([patch[int(s[0])::2, int(s[1])::2].mean() for s in sites])
    return out


def region_pass(row, rectangles):
    """Every frame of a recording reduced to its regions' mean colours, frames read in parallel."""
    def read(i):
        return region_colours(read_raw(frame_path(row.folder, i)), rectangles)
    out = np.empty((int(row.frames), len(rectangles), 3))
    with ThreadPoolExecutor(FRAME_WORKERS) as pool:
        for i, c in enumerate(pool.map(read, range(int(row.frames)))):
            out[i] = c
    return out


def cached_regions(row, rectangles):
    """One recording's region colours, from the cache when it was made from the same rectangles,
    layout and frame count."""
    signature = {'rectangles': {n: None if p is None else [p[0].start, p[0].stop, p[1].start, p[1].stop]
                                for n, p in rectangles.items()},
                 'pattern': PATTERN, 'frames': int(row.frames)}
    stem = REGION_DIR / f'{row.subject}_{row.activity}'
    meta, traces = stem.with_suffix('.json'), stem.with_suffix('.npz')
    if meta.exists() and traces.exists() and json.loads(meta.read_text()) == signature:
        with np.load(traces) as z:
            return z['colours'], False
    colours = region_pass(row, rectangles)
    REGION_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(traces, colours=colours)
    meta.write_text(json.dumps(signature))   # last: marks it complete
    return colours, True


REGION_COLOURS, RECTANGLES = {}, {}
t_start, read_now = time.time(), 0
for k, r in enumerate(recordings.itertuples(), 1):
    key = (r.subject, r.activity)
    if key not in HRV_KEYS:
        continue
    print(f'\rregion pass {k}/{len(recordings)}: {r.subject}/{r.activity}  ', end='', flush=True)
    RECTANGLES[key] = region_rectangles(key)
    REGION_COLOURS[key], fresh = cached_regions(r, RECTANGLES[key])
    read_now += fresh
print(f'\rregion pass: {len(REGION_COLOURS)} recordings, {read_now} read from the frames and '
      f'{len(REGION_COLOURS) - read_now} from the cache, in {(time.time() - t_start) / 60:.1f} min')

place_rows = []
for key, colours in REGION_COLOURS.items():
    app = TRACES[key]
    for k, (name, place) in enumerate(RECTANGLES[key].items()):
        c = colours[:, k]
        place_rows.append({'subject': key[0], 'activity': key[1], 'region': name,
                           'cells': 0 if place is None else
                           (place[0].stop - place[0].start) * (place[1].stop - place[1].start) // 4,
                           'green': np.median(c[:, 1]), 'r_g': np.median(c[:, 0] / c[:, 1]),
                           'b_g': np.median(c[:, 2] / c[:, 1]),
                           'r_g_app': np.median(app[:, 0] / app[:, 1]),
                           'b_g_app': np.median(app[:, 2] / app[:, 1])})
placement = pd.DataFrame(place_rows)
placement['chroma_gap'] = np.maximum((placement.r_g / placement.r_g_app - 1).abs(),
                                     (placement.b_g / placement.b_g_app - 1).abs())
print(f'regions not read (fewer than {MIN_REGION_CELLS} cells inside the frame): '
      f'{int((placement.cells == 0).sum())}')
print(placement.groupby('region', sort=False)
      .agg(cells=('cells', 'median'), cells_least=('cells', 'min'), green=('green', 'median'),
           green_least=('green', 'min'), r_g=('r_g', 'median'), b_g=('b_g', 'median'),
           chroma_gap=('chroma_gap', 'median'), chroma_gap_worst=('chroma_gap', 'max'))
      .round(3).to_string())
print(f'the app\'s crop, for comparison: R/G {placement.r_g_app.median():.3f}, B/G '
      f'{placement.b_g_app.median():.3f} in the median recording')
odd = placement[placement.chroma_gap > 0.15].sort_values('chroma_gap', ascending=False)
print(f'regions whose R/G or B/G departs from the app\'s crop by more than 15%: {len(odd)}')
if len(odd):
    print(odd[['subject', 'activity', 'region', 'green', 'r_g', 'b_g', 'r_g_app', 'b_g_app']]
          .round(3).to_string(index=False))

region pass: 58 recordings, 58 read from the frames and 0 from the cache, in 10.8 min
regions not read (fewer than 1000 cells inside the frame): 0
              cells  cells_least    green  green_least    r_g    b_g  chroma_gap  chroma_gap_worst
region                                                                                            
forehead     7020.0         5060  157.254       71.845  1.036  0.662       0.029             0.142
cheek_left   4650.0         3306  132.415       62.094  1.063  0.648       0.027             0.124
cheek_right  4650.5         3300  134.861       77.050  1.074  0.648       0.031             0.118
the app's crop, for comparison: R/G 1.050, B/G 0.662 in the median recording
regions whose R/G or B/G departs from the app's crop by more than 15%: 0


## 21. Region beats and the three-cornered hat
Each region's pulse is the app's POS, inverted, and its beats are section 13's steepest rises
under the same rules. The beats are gathered into heartbeats: each region's constant delay
against the region with the most beats is taken out first - a cheek's pulse can arrive a few
milliseconds before or after the forehead's - and a heartbeat collects at most one beat per
region within 150 ms of its earliest. A heartbeat needs two regions.

Over consecutive heartbeats that all three regions found, the difference between two regions'
intervals cancels everything the regions share - the heart's own variability, the pulse's
travel to the face, light and motion common to the whole face - and keeps both regions'
independent errors: `var(I_a - I_b) = V_a + V_b`. Three pairs give three such sums, solved for
each region's V. This is the three-cornered hat, used to compare clocks when none of them is a
reference. It is applied to the intervals, which SDNN feels, and separately to their
successive differences, which RMSSD feels, so that neither correction assumes the jitter is
white; it is solved per span, as an app would solve it on its capture, and reported as white
jitter, the square root of V / 2. An interval in which one region is more than 25% from the
median of the three is left out, as part 1's rules would drop it, and a negative solution,
which sampling noise produces when one region is far better than the others, is taken as zero
and counted. The two hats agree for white jitter, `V_diff = 3 V_ibi`, and their ratio is
printed as a check.

Each region's jitter against the finger, from its own RMSSD excess, stands beside the hat's:
it bounds the region's independent jitter from above, and the gap between the two is what the
region shares with the others.

In [39]:
REGION_NAMES = list(REGIONS)
PAIRS = ((0, 1), (0, 2), (1, 2))


def region_beats(key):
    """Each region's beats on the PC clock, by the app's POS and section 13's fiducial and rules;
    none for a region that was not read."""
    fps, colours = PASS[key]['fps'], REGION_COLOURS[key]
    return [np.array([]) if np.isnan(colours[:, k]).any() else
            on_pc_clock(key, camera_beats(camera_pulse(colours[:, k], fps), fps))
            for k in range(colours.shape[1])]


def heartbeats(beats):
    """One row per heartbeat, one column per region: that region's beat for it, NaN where it has none.
    Each region's constant delay against the region with the most beats is taken out first, so that
    a cheek's earlier or later pulse cannot split a heartbeat. A heartbeat takes at most one beat per
    region within MATCH_TOL_S of its earliest, and is kept where at least two regions found it."""
    base = beats[int(np.argmax([b.size for b in beats]))]
    delays = []
    for b in beats:
        gap = nearest_gap(b, base) if b.size > 1 else np.array([])
        close = gap[np.abs(gap) <= MATCH_TOL_S]
        delays.append(float(np.median(close)) if close.size else np.nan)
    events = sorted((t - d, r) for r, (b, d) in enumerate(zip(beats, delays)) if np.isfinite(d)
                    for t in b)
    rows, row, first = [], None, 0.0
    for t, r in events:
        if row is None or t - first > MATCH_TOL_S or np.isfinite(row[r]):
            if row is not None:
                rows.append(row)
            row, first = np.full(len(beats), np.nan), t
        row[r] = t
    if row is not None:
        rows.append(row)
    table = np.array(rows).reshape(-1, len(beats))
    return table[np.isfinite(table).sum(axis=1) >= 2]


def three_cornered_hat(rows, t0, t1):
    """Each region's error variance (ms²) in its intervals and in their successive differences, over
    the consecutive heartbeats that all three regions found, the intervals ending inside [t0, t1].
    The difference of two regions' series carries both regions' errors and nothing they share,
    var(a - b) = V_a + V_b, and three pairs give three such sums, solved for each region. Intervals
    where one region is more than IBI_LOCAL_TOL from the median of the three are left out, as part
    1's rules would drop them. Also returns the intervals used and whether a solution came out
    negative - sampling noise does that when one region is far better than the others; it is then
    taken as zero. Unsolved, NaN, where fewer than MIN_SPAN_INTERVALS intervals qualify."""
    none = (np.full(3, np.nan), np.full(3, np.nan), 0, False)
    full = rows[np.isfinite(rows).all(axis=1)]
    if full.shape[0] < 3:
        return none
    ibi = np.diff(full, axis=0)
    typical = np.median(ibi, axis=1, keepdims=True)
    end = np.median(full[1:], axis=1)
    use = (np.all(np.abs(ibi - typical) <= IBI_LOCAL_TOL * typical, axis=1)
           & (end >= t0) & (end <= t1))
    successive = use[1:] & use[:-1]
    if use.sum() < MIN_SPAN_INTERVALS or successive.sum() < MIN_SPAN_INTERVALS:
        return none

    def solve(series):
        v01, v02, v12 = (np.var(series[:, a] - series[:, b], ddof=1) for a, b in PAIRS)
        return 1e6 * np.array([v01 + v02 - v12, v01 + v12 - v02, v02 + v12 - v01]) / 2

    v_ibi, v_diff = solve(ibi[use]), solve(np.diff(ibi, axis=0)[successive])
    negative = bool((v_ibi < 0).any() or (v_diff < 0).any())
    return np.clip(v_ibi, 0, None), np.clip(v_diff, 0, None), int(use.sum()), negative


REGION_BEATS, HEARTBEATS, HAT, hat_rows = {}, {}, {}, []
for key in HRV_KEYS:
    REGION_BEATS[key] = region_beats(key)
    HEARTBEATS[key] = heartbeats(REGION_BEATS[key])
    rows = HEARTBEATS[key]
    f_end, f_ibi, f_kept = finger_intervals(key)
    region_ibi = [beat_intervals(b) for b in REGION_BEATS[key]]
    at = np.nanmedian(rows, axis=1)
    for span, (t0, t1) in span_bounds(key).items():
        HAT[key, span] = three_cornered_hat(rows, t0, t1)
        v_ibi, v_diff, n, negative = HAT[key, span]
        _, rmssd_f, n_f = span_hrv(f_end, f_ibi, f_kept, t0, t1)
        found = rows[(at >= t0) & (at <= t1)]
        out = {'subject': key[0], 'activity': key[1], 'span': span, 'hat_intervals': n,
               'negative': negative, 'n_finger': n_f,
               'all_three': np.isfinite(found).all(axis=1).mean() if len(found) else np.nan}
        for k, name in enumerate(REGION_NAMES):
            _, rmssd_r, n_r = span_hrv(*region_ibi[k], t0, t1)
            out[f'hat_{name}'] = np.sqrt(v_ibi[k] / 2)   # as white jitter: 2s² in the intervals
            out[f'white_{name}'] = v_diff[k] / (3 * v_ibi[k]) if v_ibi[k] > 0 else np.nan
            out[f'implied_{name}'] = np.sqrt(np.clip(rmssd_r ** 2 - rmssd_f ** 2, 0, None) / 6)
            out[f'kept_{name}'] = n_r / max(n_f, 1)
        hat_rows.append(out)
hat_table = pd.DataFrame(hat_rows)
hat_table['scored'] = (hat_table.hat_intervals > 0) & (hat_table.n_finger >= MIN_SPAN_INTERVALS)
hat_captures = hat_table[hat_table.scored & (hat_table.span != 'whole')]
app_implied = captures.set_index(['subject', 'activity', 'span']).jitter_rmssd

print(f'heartbeats found by all three regions, median 60 s capture: '
      f'{hat_table[hat_table.span != "whole"].all_three.median():.0%} | the hat solved in '
      f'{len(hat_captures)} of {int((hat_table.span != "whole").sum())} captures and '
      f'{int((hat_table.scored & (hat_table.span == "whole")).sum())} of {len(HRV_KEYS)} whole '
      f'recordings; a solution came out negative in {int(hat_captures.negative.sum())} captures')
print(f'each region\'s kept intervals per finger interval, median 60 s capture: '
      + ', '.join(f'{n} {hat_table[hat_table.span != "whole"][f"kept_{n}"].median():.2f}'
                  for n in REGION_NAMES))
print()
print('independent jitter by the hat against each region\'s jitter the finger implies (from its '
      'RMSSD excess), 60 s captures, median ms; the app\'s crop for comparison:')
columns = [f'{c}_{n}' for n in REGION_NAMES for c in ('hat', 'implied')]
table = hat_captures.groupby('activity')[columns].median().reindex(list(ACTIVITIES))
table['app_implied'] = app_implied.groupby(level='activity').median()
table.loc['all'] = [*hat_captures[columns].median(), app_implied.median()]
print(table.round(1).to_string())
print('successive differences against intervals, V_diff / 3 V_ibi (1 for white jitter), median '
      'capture: ' + ', '.join(f'{n} {hat_captures[f"white_{n}"].median():.2f}' for n in REGION_NAMES))
rho = {n: spearmanr(hat_captures[f'hat_{n}'], hat_captures[f'implied_{n}'], nan_policy='omit')[0]
       for n in REGION_NAMES}
print('Spearman of the hat with the jitter the finger implies, per region: '
      + ', '.join(f'{n} {r:+.2f}' for n, r in rho.items()))

heartbeats found by all three regions, median 60 s capture: 97% | the hat solved in 110 of 116 captures and 57 of 58 whole recordings; a solution came out negative in 5 captures
each region's kept intervals per finger interval, median 60 s capture: forehead 1.00, cheek_left 0.99, cheek_right 0.99

independent jitter by the hat against each region's jitter the finger implies (from its RMSSD excess), 60 s captures, median ms; the app's crop for comparison:
            hat_forehead  implied_forehead  hat_cheek_left  implied_cheek_left  hat_cheek_right  implied_cheek_right  app_implied
activity                                                                                                                         
stationary          10.0               8.3            13.4                12.2             11.5                 13.2         14.2
reading             16.0              15.0            13.9                17.4             14.5                 14.6         18.3
watching            1

## 22. Fused beats: the corrections compared
Each heartbeat is timed by its regions' beats averaged with weights 1/V, over the regions that
found it. So weighted, the average keeps the least of their independent errors, Σw²V / (Σw)²
per heartbeat, and over a span that adds a known amount to SDNN² and to RMSSD²: the mean, over
the span's intervals, of what their beats add. The fused beats are taken through part 1's
rules and set against the finger as in section 18, as they are and with that amount taken out.
No region is taken as timed closer than 5 ms, half the closest timing of part 2, so that a
region whose solution came out near zero cannot take all the weight.

Four estimates are compared on the spans where all four exist: the camera as section 18 read
it, the predicted correction of section 19, the fused regions, and the fused regions with the
hat's correction. The last table sets the jitter the finger implies for the app's crop and for
the fused beats beside what the hat attributes to the regions' independent errors. Where the
fused beats' implied jitter stays well above the hat's, the rest is shared by the three
regions, and no fusion of regions of one face removes it.

In [40]:
def fused_beats(rows, v_ibi, v_diff):
    """Each heartbeat's time as its regions' beats averaged with weights 1/V - V each region's interval
    error variance from the hat - over the regions that found it. Also returns what each average keeps
    of the regions' independent errors, per heartbeat and as white jitter (s²): from the intervals'
    variances, which SDNN feels, and from their successive differences', which RMSSD feels."""
    w = np.where(np.isfinite(rows), 1.0 / v_ibi, 0.0)
    share = w / w.sum(axis=1, keepdims=True)
    return ((share * np.nan_to_num(rows)).sum(axis=1), (share ** 2 * v_ibi / 2).sum(axis=1) / 1e6,
            (share ** 2 * v_diff / 6).sum(axis=1) / 1e6)


def fused_hrv(times, v_sdnn, v_rmssd, t0, t1):
    """SDNN and RMSSD of the fused beats over [t0, t1], as span_hrv computes them, and the independent
    error variance each still holds (ms²): the mean over the same intervals of what their beats add -
    v_start + v_end to an interval, v_i + 4 v_i+1 + v_i+2 to a successive difference."""
    ends, ibi, kept = beat_intervals(times)
    sdnn, rmssd, n = span_hrv(ends, ibi, kept, t0, t1)
    k = kept & (ends >= t0) & (ends <= t1)
    pair = k[1:] & k[:-1]
    in_sdnn = 1e6 * np.mean((v_sdnn[:-1] + v_sdnn[1:])[k]) if k.any() else np.nan
    in_rmssd = (1e6 * np.mean((v_rmssd[:-2] + 4 * v_rmssd[1:-1] + v_rmssd[2:])[pair])
                if pair.any() else np.nan)
    return sdnn, rmssd, n, in_sdnn, in_rmssd


fused_rows = []
for key in HRV_KEYS:
    for span, (t0, t1) in span_bounds(key).items():
        v_ibi, v_diff, n, _ = HAT[key, span]
        out = {'subject': key[0], 'activity': key[1], 'span': span}
        if n > 0:
            # No region is taken as timed closer than JITTER_FLOOR_MS of white jitter.
            v_ibi = np.maximum(v_ibi, 2 * JITTER_FLOOR_MS ** 2)
            v_diff = np.maximum(v_diff, 6 * JITTER_FLOOR_MS ** 2)
            sdnn, rmssd, n_fused, in_sdnn, in_rmssd = fused_hrv(
                *fused_beats(HEARTBEATS[key], v_ibi, v_diff), t0, t1)
            out.update({'n_fused': n_fused, 'sdnn_fused': sdnn, 'rmssd_fused': rmssd,
                        'sdnn_fused_hat': corrected(sdnn, in_sdnn),
                        'rmssd_fused_hat': corrected(rmssd, in_rmssd),
                        'hat_jitter': np.sqrt(in_rmssd / 6)})
        fused_rows.append(out)
compare = span_table.merge(pd.DataFrame(fused_rows), on=['subject', 'activity', 'span'], how='left')
compare = compare[compare.scored & (compare.n_fused >= MIN_SPAN_INTERVALS)].copy()
excess = compare.rmssd_fused ** 2 - compare.rmssd_finger ** 2
compare['fused_implied'] = np.sqrt(np.clip(excess, 0, None) / 6)
METHODS = {'camera (app crop)': 'cam', 'predicted correction': 'quality', 'fused regions': 'fused',
           'fused, hat correction': 'fused_hat'}
cap = compare[compare.capture]
whole = compare[~compare.capture]
print(f'spans with every method: {len(cap)} of {int((span_table.scored & span_table.capture).sum())} '
      f'scored 60 s captures, {len(whole)} of {int((span_table.scored & ~span_table.capture).sum())} '
      f'whole recordings')
for label, df in (('60 s captures', cap), ('whole recordings', whole),
                  ('60 s captures through the gate', cap[cap.gated])):
    print()
    print(f'{label} ({len(df)}):')
    for metric in ('sdnn', 'rmssd'):
        print(f'{metric.upper()}:')
        print(pd.DataFrame({m: error_scores(df[f'{metric}_{c}'], df[f'{metric}_finger'])
                            for m, c in METHODS.items()}).T.drop(columns='spans').round(3).to_string())
print()
for metric, column in (('sdnn', 'within_10'), ('sdnn', 'median'), ('rmssd', 'median')):
    print(f'60 s captures by activity, {metric.upper()} {column.replace("_", " ")}:')
    print(pd.DataFrame({m: by_activity(cap, f'{metric}_{c}', f'{metric}_finger')[column]
                        for m, c in METHODS.items()}).round(3).to_string())
print()
shared = cap.groupby('activity')[['jitter_rmssd', 'fused_implied', 'hat_jitter']].median()
shared.loc['all'] = cap[['jitter_rmssd', 'fused_implied', 'hat_jitter']].median()
shared.columns = ['app crop, implied', 'fused, implied', 'fused, by the hat']
print('jitter (ms, as white jitter from the RMSSD excess) of the app\'s crop and of the fused beats '
      'against the finger, and what the hat attributes to the regions\' independent errors, median '
      '60 s capture:')
print(shared.reindex([*ACTIVITIES, 'all']).round(1).to_string())

spans with every method: 110 of 116 scored 60 s captures, 57 of 58 whole recordings

60 s captures (110):
SDNN:
                       median  median_abs  within_10  within_20  worst
camera (app crop)       0.142       0.142      0.418      0.591  4.209
predicted correction   -0.001       0.097      0.527      0.755  2.469
fused regions           0.030       0.044      0.664      0.773  1.401
fused, hat correction   0.009       0.042      0.727      0.818  1.246
RMSSD:
                       median  median_abs  within_10  within_20  worst
camera (app crop)       0.624       0.624      0.118      0.264  7.818
predicted correction    0.108       0.293      0.155      0.391  4.579
fused regions           0.128       0.132      0.427      0.582  4.625
fused, hat correction   0.064       0.093      0.527      0.618  4.137

whole recordings (57):
SDNN:
                       median  median_abs  within_10  within_20  worst
camera (app crop)       0.131       0.131      0.439      0.596  3.442

## 23. Findings and decision (part 3)

**Picking the beats is not what costs the camera its HRV.** Section 15 scored the camera on
the finger's beats; here it keeps or drops its own intervals under part 1's rules, with no
reference involved, and the result barely moves: RMSSD runs 56% above the finger over whole
recordings against the 54% of part 2, and SDNN 13%. Over the 60 s span an app records, SDNN is
16% high and RMSSD 63% high in the median capture, with SDNN within 10% of the finger in 40%
of captures and RMSSD in 11%. By activity the order is the one part 2 set: SDNN 4% high in
deep breathing and 8% at rest, 55% while talking. Whatever stands between this camera and
usable HRV is the timing of the beats it finds, not which beats it finds.

**The excess is what independent jitter predicts, from both ends.** The jitter implied by the
SDNN excess and by the RMSSD excess agree in the median capture - 21.1 against 21.2 ms, with a
Spearman correlation of +0.94 between them across captures - as they only do if the error is
independent of the beat times it perturbs. Per activity it runs 14.2 ms at rest, 17.3 ms in
deep breathing, 18.3 ms reading, 22.4 ms watching and 33.9 ms talking.

**The capture's own quality numbers see the jitter.** The app's window confidence correlates
-0.65 with it, the share of intervals the rules drop +0.70, and the beats' correlation with
the recording's median beat shape -0.61. A gate at confidence 0.80 and 2% dropped keeps 73 of
116 captures - 21 stationary, 22 deep, 16 reading, 12 watching and 2 of 26 talking - and
those captures are 7.5% high on SDNN, within 10% in 56%.

**Predicting the jitter corrects SDNN and stalls on RMSSD.** Fitted on the other thirteen
subjects' captures, the three quality numbers together bring SDNN's median absolute error from
16.2% to 10.0% and the share within 10% from 40% to 51%, against a ceiling of 4.2% and 75% for
a correction that uses the finger's own jitter; confidence alone reaches 10.7%. On RMSSD the
same correction reaches only 29.5%, and 43 of the 116 corrections hit the floor that forbids
taking more than half the value - the floor, not the fit, is what that number measures, since
6s² of a 21 ms jitter is most of a 43 ms RMSSD. Predicted and implied jitter correlate +0.56.
The fitted coefficients are not interpretable: confidence and the dropped share carry the same
information, and the sign on confidence flips between the correlation and the fit.

**Three regions of one face disagree only by their own errors, and the hat separates them.**
The forehead and both cheeks sit on skin by the data - R/G 1.04-1.07 and B/G 0.65-0.66 against
the app's crop at 1.05 and 0.66, none more than 15% away, 3,300 cells at least - and all three
find 97% of the heartbeats in the median capture. The three-cornered hat solves in 110 of 116
captures, negative in 5, and its estimates stand up to the two checks available: each region's
independent jitter by the hat lands within 1-3 ms of what that region's own RMSSD excess
implies against the finger (forehead 12.9 against 13.3 ms, cheeks 14.7 against 17.3 and 14.3
against 15.2), and the hat solved on successive differences agrees with the hat solved on
intervals to 0.97-0.98 of the white-jitter ratio. Each region alone is already timed more
closely than the app's single crop, 13-15 ms against 21.2 ms.

**Fusing the regions halves the jitter and brings RMSSD within reach.** Weighted by 1/V, the
fused beats are timed to 9.7 ms against the finger in the median capture, from 20.0 ms for the
app's crop, and 7.2 ms at rest - the 7 ms that a 40 ms resting RMSSD tolerates. Fused and
corrected by the hat, SDNN is 0.9% high in the median capture and within 10% in 73% of them,
and RMSSD 6.4% high and within 10% in 53%; through the gate, 90% and 73%. Over whole
recordings the same two land at 0.6% and 3.9%. Fusion alone accounts for most of this, the
hat's correction for the rest, and both beat the predicted correction on RMSSD by a factor
of three in median absolute error.

**What the regions share sets the floor.** The hat attributes 6.9 ms of the fused beats' 9.7
to the regions' independent errors, leaving about 7 ms that all three carry together - the
pulse's travel to the face against its travel to the finger, the light, the motion of the
whole head. No fusion of regions on one face removes it, and it is the same order as the 7 ms
a 40 ms reference tolerates, so this is where the approach stops rather than a step on the way
to arbitrary precision.

**Talking stays out of reach.** Fused and corrected it is still 25% high on SDNN and 68% on
RMSSD, with jitter of 27.7 ms against the hat's 18.7, and only 2 of its 26 captures pass the
gate. Watching, reading and deep breathing all come within 3-7% on SDNN once fused.

**Decision.**
- **Region fusion is the candidate HRV path, and it is a change to the app, not a correction
  applied afterwards.** It needs three colour traces and a hat solved per capture, so the
  readout, not the reporting, is what changes. The regions overlap the app's crop, so this
  replaces the single-crop estimate rather than corroborating it.
- **Its rule is written before the next dataset, not here.** Everything above was measured on
  the recordings it was developed on, including the gate, and 14 subjects of one camera under
  one set of lights cannot settle a threshold. The preprocessing store comes first, then a
  pre-registered rule - which activities, which gate, which statistic, and what counts as
  success - and then an independent set.
- **HRV is reported with its span and activity, never pooled across them.** A 60 s capture and
  a whole recording differ, talking is excluded, and the numbers quoted are ratios whose worst
  cases belong to references near 13 ms, where any jitter is large in proportion.
- **RMSSD is corrected by fusion, not by prediction.** The predicted correction stands for
  SDNN, where it adds half of what the finger's own jitter could; on RMSSD it hits its floor
  in 37% of captures and should not be reported as a correction at all.
- **The existing HRV harness is corrected first** (review items 3 and 17): the alignment by the
  median distance between beats of part 2, a minimum beat spacing tied to each recording's
  beat period rather than a fixed 0.3 s, a search wide enough for the offsets a dataset
  documents, and the sampling ceiling computed from the refined fiducial rather than from the
  noise RMSSD.

**What the dataset supports, after three parts.** Heart rate in all 70 recordings, where the
app's own POS reading is within 3.6 bpm of the finger. HRV in 58, where the finger's resting
RMSSD of 40 ms is high enough to be a forgiving test. A breathing probe at a paced rate that
13 of 14 subjects follow, as a target rather than a ground truth. And one measurement that
changes the earlier verdict's terms: HRV from this camera failed on timing, not on sampling or
on beat detection, and timing improves by a factor of two when the face is read as three
regions instead of one - leaving a shared residual of about 7 ms that one face cannot resolve
further.